# Figure 6A–L and Supplementary Figures S16–S17: cell-line BioCycle validation

This self-contained Python notebook uses the **current GMCF-1708 processed results** consistently across all three figures. The retained biological cell lines are ranked from the current validation metrics. Figure 6A–C add the supplied Kronos BMAL1/PER2 workbook; D–L retain the current BioCycle validation results. S16A–B add the TCGA transcriptome PCA and core-clock abundance comparison; S16J shows PANC-1 and S16K shows Panc 10.05.

**Input boundary.** BioCycle gene/event rhythmicity calls, harmonic comparison metrics, and current validation/reference tables are explicit processed upstream inputs. This notebook embeds the downstream calculations and plotting; it does not rerun sequencing quantification, the external BioCycle prediction service, or upstream ChIP/CLIP inference. Their connection to `PDAC_Rhythmicity/process.smk` is not established here. No legacy notebook, project helper, or saved figure is imported at runtime. Supply the listed processed inputs separately when sharing the notebook.

**Interpretation.** These synchronized cell-line time courses use observed numeric ZT, not CHIRAL-inferred phase. Rhythmicity is nominal *p* < 0.05 as in the current BioCycle analysis. Tumor-only cell lines do not directly establish a tumor-versus-normal difference; clock-TF results are context, and driver annotations are descriptive. Composite alignment is the existing summary of eight support/effect measures. Hs 766T is the highest-ranked retained line, not an independent selection for validation.

**Publication outputs.** Individual panels and assemblies are saved as PNG/PDF under `data/figures/publication/fig06_cellline_biocycle_validation/`; tables and input hashes are under `data/publication/fig06_cellline_biocycle_validation/`. Every required panel and assembly appears inline. Module widths are multiples of 52.5 mm; all text roles are globally configured at 7 pt or larger. PNGs omit panel letters; PDFs include them. The historical S1/S2 labeling exception does not apply to S16/S17.

Use `CHRONO_KRONOS_PATH` to override the default `data/kronos/Kronos.xlsx` workbook path.

Run all cells with the project's Docker Python 3 kernel. Set `CHRONO_PROJECT_ROOT` for the publication repository and `CHRONO_CELLLINE_DATA_DIR` for the processed CircadianMultiOmics data directory when using a different installation.

## 1. Setup and processed inputs

In [ ]:
# Cell 1: Setup, paths, global fonts, and export helpers
from pathlib import Path
import os
import sys
import json
import time
import hashlib
import platform
import importlib.metadata
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.text import Text
from scipy import stats
from IPython.display import display, Image

START_TIME = time.time()
root_candidates = [os.environ.get('CHRONO_PROJECT_ROOT', ''), str(Path.cwd()),
                   str(Path.cwd().parent),
                   str(Path(os.environ.get('WORKSPACE_ROOT', '/home/wdeng3/workspace/Codespace')) / 'PDAC_Rhythmicity')]
PROJECT_ROOT = next((Path(p).resolve() for p in root_candidates
                     if p and (Path(p) / 'notebook_publication').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Set CHRONO_PROJECT_ROOT to the publication repository root.')
CMO_DATA = Path(os.environ.get('CHRONO_CELLLINE_DATA_DIR',
                              str(PROJECT_ROOT.parent / 'CircadianMultiOmics' / 'data'))).resolve()
if not CMO_DATA.is_dir():
    raise FileNotFoundError('Set CHRONO_CELLLINE_DATA_DIR to the current cell-line processed-data directory.')
KRONOS_PATH = Path(os.environ.get('CHRONO_KRONOS_PATH',
    str(PROJECT_ROOT / 'data' / 'kronos' / 'Kronos.xlsx'))).resolve()
ANALYSIS_SLUG = 'fig06_cellline_biocycle_validation'
TABLE_DIR = PROJECT_ROOT / 'data' / 'publication' / ANALYSIS_SLUG
FIGURE_DIR = PROJECT_ROOT / 'data' / 'figures' / 'publication' / ANALYSIS_SLUG
LOG_DIR = PROJECT_ROOT / 'data' / 'logs' / 'publication' / ANALYSIS_SLUG
for directory in (TABLE_DIR, FIGURE_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RNG_SEED = 606
np.random.seed(RNG_SEED)
PANEL_MM = 210 / 4
FONT = dict(base=8, title=9, axis_title=8, tick=7, legend=7, strip=8,
            annotation=7, tag=10, subtitle=7)
FONT_FAMILY = 'DejaVu Sans'
assert min(FONT.values()) >= 7
mpl.rcParams.update({
    'font.family': FONT_FAMILY, 'font.size': FONT['base'],
    'axes.titlesize': FONT['title'], 'axes.titleweight': 'bold',
    'axes.labelsize': FONT['axis_title'], 'xtick.labelsize': FONT['tick'],
    'ytick.labelsize': FONT['tick'], 'legend.fontsize': FONT['legend'],
    'legend.title_fontsize': FONT['legend'], 'axes.linewidth': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'lines.linewidth': 0.8, 'figure.dpi': 120, 'savefig.dpi': 300,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'figure.facecolor': 'white',
    'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'savefig.bbox': None,
})
INPUT_MANIFEST = {}
EXPORT_MANIFEST = []

def register_input(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing processed input: {path}')
    if path not in INPUT_MANIFEST:
        digest = hashlib.sha256()
        with path.open('rb') as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(block)
        input_root = PROJECT_ROOT if path == KRONOS_PATH or path.resolve().is_relative_to(PROJECT_ROOT) else CMO_DATA
        INPUT_MANIFEST[path] = dict(input_root='CHRONO_PROJECT_ROOT' if input_root == PROJECT_ROOT else 'CHRONO_CELLLINE_DATA_DIR',
                                    input_path=os.path.relpath(path, input_root),
                                    bytes=path.stat().st_size, sha256=digest.hexdigest())
    return path

def read_table(path, **kwargs):
    return pd.read_csv(register_input(path), **kwargs)


def save_figure(fig, stem, width_modules, height_modules, tags=None):
    """Save fixed-size PNG without panel letters, then the labeled vector PDF."""
    assert width_modules in (1, 2, 3, 4)
    assert int(height_modules) == height_modules and height_modules >= 1
    width_mm, height_mm = PANEL_MM * width_modules, PANEL_MM * height_modules
    fig.set_size_inches(width_mm / 25.4, height_mm / 25.4, forward=True)
    fig.canvas.draw()
    text_artists = [a for a in fig.findobj(Text) if a.get_visible() and a.get_text().strip()]
    minimum = min(a.get_fontsize() for a in text_artists)
    if minimum < 7 - 1e-8:
        raise AssertionError(f'{stem}: text below 7 pt ({minimum})')
    target = FIGURE_DIR / stem
    target.parent.mkdir(parents=True, exist_ok=True)
    png, pdf = target.with_suffix('.png'), target.with_suffix('.pdf')
    fig.savefig(png, dpi=300, bbox_inches=None, facecolor='white')
    label_artists = [fig.text(x, y, label, ha='left', va='top', fontsize=FONT['tag'],
                              weight='bold', family=FONT_FAMILY)
                     for label, x, y in (tags or [])]
    fig.savefig(pdf, bbox_inches=None, facecolor='white')
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    canvas = fig.bbox
    outside = []
    for artist in fig.findobj(Text):
        if not artist.get_visible() or not artist.get_text().strip():
            continue
        box = artist.get_window_extent(renderer)
        if box.x0 < -1 or box.y0 < -1 or box.x1 > canvas.width + 1 or box.y1 > canvas.height + 1:
            outside.append(artist.get_text())
    for artist in label_artists:
        artist.remove()
    EXPORT_MANIFEST.append(dict(stem=stem, width_modules=width_modules,
        height_modules=height_modules, width_mm=width_mm, height_mm=height_mm,
        minimum_text_pt=minimum, png_has_panel_labels=False, pdf_has_panel_labels=bool(tags),
        pdf_panel_labels=','.join(tag[0] for tag in (tags or [])),
        outside_text=' | '.join(outside), png=str(png.relative_to(FIGURE_DIR)),
        pdf=str(pdf.relative_to(FIGURE_DIR))))
    display(Image(filename=str(png), width=min(1100, round(width_mm / 25.4 * 120))))
    plt.close(fig)
    if outside:
        warnings.warn(f'{stem}: inspect text outside page: {outside}')
    return png, pdf

print('Current processed cell-line inputs:', CMO_DATA)
print('Publication module (mm):', PANEL_MM)
print('Fonts (pt):', FONT)


## 2. Current data, downstream analyses, and source tables

The analysis starts from the current, processed BioCycle validation tables and BioCycle call files. It does not refit BioCycle, recreate upstream CLIP/clock-TF enrichment, or modify any legacy result. `read_table` is supplied by the publication notebook and records each input hash.

In [ ]:
# Cell 2: Figure 6/S16/S17 analysis parameters and self-contained helpers
import re
from scipy.stats import norm

PVAL = 0.05
DENSITY_BW = 0.35
DENSITY_GRID = 360
SOURCE_DIR = CMO_DATA / 'cellline_validation' / '1708_comprehensive_biocycle'
HARMONIC_DIR = CMO_DATA / 'cellline_validation' / '1708_comprehensive'
BIOCYCLE_DIR = CMO_DATA / 'biocycle_analysis'
EVENT_DIR = CMO_DATA / 'cellline_validation' / 'biocycle_events'
RBP_CATALOG_PATH = CMO_DATA / 'ref' / 'rbp_rna_processing_links' / 'rbp_catalog.tsv'

CORE_CLOCK = ['BMAL1','CLOCK','NPAS2','CRY1','CRY2','PER1','PER2','PER3','NR1D1','NR1D2','DBP','TEF','NFIL3']
MODULES = {
    'Core clock': CORE_CLOCK,
    'Cell cycle': ['MKI67','PCNA','MCM2','MCM3','MCM4','MCM5','MCM6','MCM7','CCNA2','CCNB1','CCNB2','CCND1','CCNE1','CDK1','CDK2','CDK4','CDK6','E2F1','CHEK1','PLK1','AURKA','AURKB'],
    'RAS/MAPK signaling': ['KRAS','NRAS','HRAS','BRAF','RAF1','MAP2K1','MAP2K2','MAPK1','MAPK3','DUSP4','DUSP6','FOS','JUN','MYC'],
    'RNA processing/RBP': ['ELAVL1','IGF2BP1','IGF2BP2','IGF2BP3','HNRNPA1','HNRNPC','SRSF1','SRSF3','PTBP1','RBFOX2','QKI','ESRP1','ESRP2','MBNL1','CELF1','KHDRBS1','FUS','TARDBP'],
    'APA machinery': ['CPSF1','CPSF2','CPSF3','CPSF4','CPSF6','CPSF7','CSTF1','CSTF2','CSTF3','NUDT21','CLP1','PCF11','PABPN1','PAPOLA','SYMPK','FIP1L1'],
    'Splicing machinery': ['SF3B1','SF3B2','SF3B3','SF3A1','SF3A2','U2AF1','U2AF2','SRSF1','SRSF2','SRSF3','SRSF5','SRSF6','HNRNPA1','HNRNPC','PTBP1','RBFOX2','QKI','ESRP1','ESRP2','MBNL1','MBNL2','CELF1','PRPF8','PRPF19','SNRNP70','SNRPA','SNRPB','DDX5','DDX17'],
}

def tsv(path, **kwargs):
    return read_table(path, sep='\t', **kwargs)

def circular_summary(phases):
    a = np.asarray(phases, dtype=float); a = a[np.isfinite(a)] % 24.0
    if not len(a):
        return dict(n=0, mean_phase_hr=np.nan, mean_phase_rad=np.nan, R=np.nan, axis_phase_hr=np.nan, axis_phase_rad=np.nan, opposite_axis_phase_hr=np.nan, opposite_axis_phase_rad=np.nan, R2=np.nan)
    theta = a * 2 * np.pi / 24.0
    mean_rad = float(np.arctan2(np.sin(theta).mean(), np.cos(theta).mean()) % (2*np.pi))
    z2 = np.exp(2j * theta).mean(); axis_rad = float((np.angle(z2) / 2) % np.pi); axis_hr = axis_rad * 24 / (2*np.pi)
    return dict(n=int(len(a)), mean_phase_hr=mean_rad*24/(2*np.pi), mean_phase_rad=mean_rad, R=float(abs(np.exp(1j*theta).mean())), axis_phase_hr=axis_hr, axis_phase_rad=axis_rad, opposite_axis_phase_hr=(axis_hr+12)%24, opposite_axis_phase_rad=(axis_rad+np.pi)%(2*np.pi), R2=float(abs(z2)))

def circular_kde(phases, bw=DENSITY_BW, n_grid=DENSITY_GRID):
    a = np.asarray(phases, dtype=float); a = a[np.isfinite(a)] % 24.0
    if not len(a): return np.array([]), np.array([])
    theta = a * 2*np.pi/24; grid = np.linspace(0, 2*np.pi, n_grid, endpoint=False); kappa = 1/(bw**2)
    density = np.exp(kappa*np.cos(theta[:,None]-grid[None,:])).mean(axis=0)
    density /= density.sum() * (2*np.pi/n_grid)
    return np.r_[grid*24/(2*np.pi), 24.0], np.r_[density, density[0]]

def stouffer_p(pvals):
    p = np.asarray(pvals, float); p = p[np.isfinite(p)]
    if not len(p): return np.nan
    p = np.clip(p, 1e-300, 1-1e-12)
    return float(norm.sf(norm.isf(p).sum()/np.sqrt(len(p))))

def support_from_metrics(metrics):
    m = metrics.set_index('line')
    return pd.DataFrame({
        'ExprPol': m.expr_p.lt(PVAL), 'ClockTF': m.clockTF_OR.gt(1) & m.clockTF_p.lt(PVAL),
        'SEPol': m.SE_p.lt(PVAL), 'APAamp': m.APA_med_amp.gt(0.05), 'APAlag': m.APAlag_p.lt(PVAL),
        'CLIP_APA': m.clipAPA_nRBP.ge(5), 'RBPpol': m.RBP_p.lt(PVAL), 'CoreClk': m.n_coreclock.ge(2),
    }).fillna(False).astype(bool)

def alignment_rank(metrics, supports):
    effect = ['expr_R2','clockTF_OR','SE_R2','APA_med_amp','APAlag_R2','clipAPA_nRBP','RBP_R2','n_coreclock']
    out = metrics.set_index('line')[['cell','series_label']].copy()
    out['n_supported'] = supports.sum(axis=1).reindex(out.index).astype(int)
    out['effect_rankpct'] = metrics.set_index('line')[effect].rank(pct=True).mean(axis=1).fillna(0)
    out['composite'] = out.n_supported + out.effect_rankpct
    return out.reset_index().sort_values(['composite','n_supported','effect_rankpct'], ascending=False, kind='stable').reset_index(drop=True)

def write_source_table(frame, name):
    path = TABLE_DIR / name; frame.to_csv(path, sep='\t', index=False); return path


In [ ]:
# Cell 3: Load current validation outputs and recompute downstream support and rank
metrics = tsv(SOURCE_DIR / 'perline_metric_matrix.tsv')
metrics['plot_label'] = metrics['cell']
overall = tsv(SOURCE_DIR / 'overall_alignment_to_cohort.tsv')
drivers = tsv(SOURCE_DIR / 'driver_panel_with_metrics.tsv')
harmonic_metrics = tsv(HARMONIC_DIR / 'perline_metric_matrix.tsv')

required = {'line','cell','series_label','expr_R2','expr_p','clockTF_OR','clockTF_p','SE_R2','SE_p','APA_med_amp','APAlag_R2','APAlag_p','clipAPA_nRBP','RBP_R2','RBP_p','n_coreclock'}
missing = required - set(metrics.columns)
assert not missing, f'Missing required current BioCycle metric columns: {sorted(missing)}'
assert len(metrics) == 10, f'Expected 10 retained lines, found {len(metrics)}'
assert 'CAPAN1_Dexa' in set(metrics.line) and 'CAPAN1_Dx' not in set(metrics.line)
expected_counts = {'HS766T_Dexa': 2151, 'PANC1_Dx': 821, 'PANC1005_Dexa': 3529}
for line, expected in expected_counts.items():
    observed = int(metrics.loc[metrics.line.eq(line), 'n_rhythmic_genes'].iloc[0])
    assert observed == expected, f'{line}: expected {expected:,}, got {observed:,}'

supports = support_from_metrics(metrics)
rank = alignment_rank(metrics, supports)
rank['plot_label'] = rank.line.map(metrics.set_index('line').cell)
label_order = rank.line.tolist(); label_map = metrics.set_index('line').cell.to_dict()
ASPECTS = {'Core oscillator retained':'n_coreclock','Rhythmic-gene polarization':'expr_R2','Clock-TF enrichment':'clockTF_OR','Rhythmic-SE polarization':'SE_R2','APA-mRNA lag concentration':'APAlag_R2','RBP acrophase organization':'RBP_R2','CLIP-target APA enrichment':'clipAPA_nRBP'}
best_per_aspect = pd.DataFrame([{'aspect': aspect, 'line': (sub := metrics.dropna(subset=[col])).loc[sub[col].idxmax(), 'line'], 'cell': sub.loc[sub[col].idxmax(), 'series_label'], 'value': float(sub[col].max())} for aspect, col in ASPECTS.items()])

# The overall alignment and driver annotations are current processed validation outputs.
# They retain upstream cohort/CLIP/clock-TF provenance; their downstream plotting order is recomputed above.
harmonic = harmonic_metrics[harmonic_metrics.line.isin(metrics.line)].copy()

def method_meta(frame, method):
    sup = support_from_metrics(frame); rows=[]
    for claim, effect, pcol, scol in [('Rhythmic-gene polarization','expr_R2','expr_p','ExprPol'),('Clock-TF coupling (OR)','clockTF_OR','clockTF_p','ClockTF'),('Rhythmic-SE polarization','SE_R2','SE_p','SEPol'),('APA-mRNA lag concentration','APAlag_R2','APAlag_p','APAlag'),('RBP acrophase organization','RBP_R2','RBP_p','RBPpol')]:
        s=frame.dropna(subset=[pcol]); n=len(s); ns=int(sup.loc[s.line,scol].sum()) if n else 0; p=stouffer_p(s[pcol])
        verdict='Strong' if n and p<PVAL and ns/n>=.75 else ('Significant' if n and p<PVAL else ('Partial' if n and ns/n>=.5 else 'Weak')) if n else 'not tested'
        rows.append({'method':method,'claim':claim,'n_lines':n,'n_sig':ns,'support_fraction':ns/n if n else np.nan,'mean':s[effect].mean(),'combined_p':p,'verdict':verdict})
    for claim, effect in [('APA rhythmic amplitude','APA_med_amp'),('CLIP-target APA enrichment (#RBP)','clipAPA_nRBP'),('Core-clock genes rhythmic (#)','n_coreclock')]:
        s=frame.dropna(subset=[effect]); rows.append({'method':method,'claim':claim,'n_lines':len(s),'n_sig':np.nan,'support_fraction':np.nan,'mean':s[effect].mean(),'combined_p':np.nan,'verdict':'descriptive'})
    return pd.DataFrame(rows)

hmeta, bmeta = method_meta(harmonic, 'Harmonic'), method_meta(metrics, 'BioCycle')
method_comparison = hmeta.merge(bmeta, on='claim', suffixes=('_harmonic','_biocycle'), how='outer')
method_comparison['robust_after_filtering'] = np.where(method_comparison.verdict_harmonic.isin(['Strong','Significant','descriptive']) & method_comparison.verdict_biocycle.isin(['Strong','Significant','descriptive']), 'YES', 'PARTIAL')

assert label_order[:3] == ['HS766T_Dexa','PANC1_Dx','PANC1005_Dexa'], label_order[:3]
assert int(supports.loc['HS766T_Dexa'].sum()) == 7
print('Retained lines:', len(metrics), '| top ranked:', ', '.join(label_map[x] for x in label_order[:3]))


In [ ]:
# Cell 4: Cell 4.2: Check BioCycle gene identifiers before module and clock summaries
# Module and core-clock summaries require one current BioCycle row per gene within a line.
_gene_qc = read_table(BIOCYCLE_DIR / 'biocycle_all_results.csv', usecols=['gene', 'cell_line'])
_gene_qc = _gene_qc[_gene_qc.cell_line.isin(metrics.line)].dropna(subset=['gene']).copy()
_gene_qc['gene'] = _gene_qc['gene'].astype(str)
_duplicate_gene_counts = _gene_qc.duplicated(['cell_line', 'gene']).groupby(_gene_qc.cell_line).sum()
assert int(_duplicate_gene_counts.sum()) == 0, _duplicate_gene_counts[_duplicate_gene_counts.gt(0)].to_dict()


In [ ]:
# Cell 5: Recompute phase, core-clock, module, and heatmap plot sources
# Select the canonical BioCycle columns explicitly: the source also carries a later q_value column.
expr_all = read_table(BIOCYCLE_DIR / 'biocycle_all_results.csv',
    usecols=['gene', 'cell_line', 'P_VALUE', 'Q_VALUE', 'PERIOD', 'LAG', 'AMPLITUDE', 'OFFSET'])
se_all = read_table(EVENT_DIR / 'biocycle_se_all.csv')
rbp_catalog = tsv(RBP_CATALOG_PATH)
rbps = set(rbp_catalog.gene_symbol.dropna().astype(str))
expr_all = expr_all.rename(columns={'P_VALUE':'p_value','Q_VALUE':'q_value','LAG':'acrophase','AMPLITUDE':'amplitude','PERIOD':'period','OFFSET':'mesor'})
assert {'gene','cell_line','p_value','acrophase','amplitude'}.issubset(expr_all.columns)

def expression_for(line, rhythmic=False):
    d = expr_all.loc[expr_all.cell_line.eq(line)].copy()
    return d.loc[d.p_value.lt(PVAL)].copy() if rhythmic else d

def apa_for(line):
    d = read_table(EVENT_DIR / f'biocycle_apa_{line}.csv')
    if 'is_rhythmic' not in d: d['is_rhythmic'] = d.p_value.lt(PVAL)
    return d

def phases_for(line, layer):
    if layer == 'Expression': return expression_for(line, True).acrophase.to_numpy(float)
    if layer == 'Skipped exon': return se_all.loc[se_all.CELL_LINE.eq(line) & se_all.P_VALUE.lt(PVAL), 'LAG'].to_numpy(float)
    if layer == 'APA': return apa_for(line).query('is_rhythmic').acrophase.to_numpy(float)
    if layer == 'RBP expression':
        x=expression_for(line, True); return x.loc[x.gene.isin(rbps), 'acrophase'].to_numpy(float)
    raise ValueError(layer)

phase_stats_rows=[]; phase_density_rows=[]; phase_value_rows=[]
for line in label_order:
    for layer in ['Expression','Skipped exon','APA','RBP expression']:
        phases = phases_for(line, layer); summary = circular_summary(phases)
        base={'line':line,'cell':label_map[line],'plot_label':label_map[line],'layer':layer,'bw':DENSITY_BW}
        phase_stats_rows.append({**base, **summary})
        phase_value_rows.extend({**base,'phase_hr':float(x%24)} for x in np.asarray(phases,float) if np.isfinite(x))
        if summary['n'] >= 3:
            hours, density = circular_kde(phases)
            phase_density_rows.extend({**base,'phase_hr':float(h),'theta':float(h*2*np.pi/24),'density':float(d)} for h,d in zip(hours,density))
phase_stats=pd.DataFrame(phase_stats_rows); phase_density=pd.DataFrame(phase_density_rows); phase_values=pd.DataFrame(phase_value_rows)

core_rows=[]; module_rows=[]
for line in label_order:
    x = expression_for(line, False).dropna(subset=['gene']).copy(); x.gene=x.gene.astype(str).replace({'ARNTL':'BMAL1'})
    assert x.gene.is_unique, f'Duplicate BioCycle gene identifiers in {line}'
    x=x.set_index('gene', drop=False)
    for gene in CORE_CLOCK:
        row=x.loc[gene] if gene in x.index else None
        core_rows.append({'line':line,'cell':label_map[line],'plot_label':label_map[line],'gene':gene,'p_value':float(row.p_value) if row is not None else np.nan,'q_value':float(row.q_value) if row is not None and 'q_value' in row else np.nan,'acrophase':float(row.acrophase%24) if row is not None and pd.notna(row.acrophase) else np.nan,'amplitude':float(row.amplitude) if row is not None else np.nan,'period':float(row.period) if row is not None and pd.notna(row.period) else np.nan,'is_rhythmic':bool(row is not None and row.p_value<PVAL)})
    for module, genes in MODULES.items():
        present=[g for g in genes if g in x.index]; rhythmic=x.loc[present].query('p_value < @PVAL') if present else x.iloc[0:0]
        cs=circular_summary(rhythmic.acrophase.to_numpy(float) if len(rhythmic) else [])
        module_rows.append({'line':line,'cell':label_map[line],'plot_label':label_map[line],'module':module,'n_present':len(present),'n_rhythmic':len(rhythmic),'frac_rhythmic':len(rhythmic)/len(present) if present else np.nan,'median_amplitude':rhythmic.amplitude.median() if len(rhythmic) else np.nan,'mean_phase_hr':cs['mean_phase_hr'],'phase_R':cs['R'],'axis_phase_hr':cs['axis_phase_hr'],'phase_R2':cs['R2']})
core = pd.DataFrame(core_rows); module_table = pd.DataFrame(module_rows)

def input_matrix(line):
    raw=tsv(BIOCYCLE_DIR/'inputs'/f'input_{line}.txt'); raw=raw.rename(columns={raw.columns[0]:'symbol'})
    cols=[]; times=[]
    for col in raw.columns[1:]:
        hit=re.search(r'ZT(\d+)', str(col));
        if hit: cols.append(col); times.append(int(hit.group(1)))
    a=raw.set_index('symbol')[cols].apply(pd.to_numeric,errors='coerce'); a.columns=times
    return a.T.groupby(level=0).mean().T.groupby(level=0).mean().reindex(sorted(set(times)),axis=1)

def one_heatmap(line):
    matrix=input_matrix(line); rhythmic=expression_for(line,True).dropna(subset=['gene','acrophase']).copy(); rhythmic.gene=rhythmic.gene.astype(str)
    rhythmic=rhythmic[rhythmic.gene.isin(matrix.index)].sort_values(['acrophase','p_value','gene']).drop_duplicates('gene')
    raw=matrix.loc[rhythmic.gene]; centered=raw.sub(raw.mean(axis=1),axis=0); z=centered.div(centered.std(axis=1,ddof=0).replace(0,np.nan),axis=0).replace([np.inf,-np.inf],np.nan).fillna(0)
    meta=rhythmic[[c for c in ['gene','p_value','q_value','acrophase','amplitude','period','mesor'] if c in rhythmic]].copy(); meta.insert(0,'row_order',np.arange(1,len(meta)+1)); meta.insert(1,'line',line); meta.insert(2,'cell',label_map[line])
    return {'line':line,'cell':label_map[line],'z':z,'metadata':meta,'times':np.asarray(z.columns,float)}

heatmap_lines=[x for x in label_order if not x.startswith('CAPAN1_')][:3]
heatmaps={line:one_heatmap(line) for line in heatmap_lines}
assert heatmaps['HS766T_Dexa']['z'].shape[0] == 2151


### Driver-gene reference audit (23 September 2026)

S16F uses the 40 gene-by-cell-line annotations below. **WT** means explicitly reported wild type in the cited reference; **mut** means a reported sequence variant (including small indels and splice variants); **del** means a reported gene-level or multiexon deletion; **nd** means unresolved. An omitted database entry alone never establishes WT. These are reference annotations, not resequencing of the exact stocks used in the synchronized time courses. Mutation labels do not by themselves assert pathogenicity or functional loss. Zygosity remains source-specific in the evidence table and is not displayed.

The audit corrects the Capan-2 TP53 and CDKN2A WT calls, adds missing SMAD4 annotations, and retains unresolved calls where evidence is incomplete or conflicting. In particular, Panc 03.27 CDKN2A copy-number reports conflict historically; an explicit ATCC deletion record now supports the reference deletion call. Panc 10.05 CDKN2A is explicitly reported WT in primary studies; low transcript abundance alone does not establish a genomic alteration. Its SMAD4 annotation remains unresolved because published deletion and WT descriptions conflict. The source table preserves every original call alongside its replacement, variant, URL, PMID/DOI and qualification. Current circadian metrics and ranking are unaffected.

Key primary sources and curated records:

- [TP53 cell-line reassessment](https://pmc.ncbi.nlm.nih.gov/articles/PMC4451114/): Capan-2 c.375G>T disrupts splicing despite its synonymous protein notation.
- [Okamoto et al., 1994](https://doi.org/10.1073/pnas.91.23.11045) and [Capan-2 Cellosaurus record](https://www.cellosaurus.org/CVCL_0026): CDKN2A duplication.
- [Suzuki et al., 2008](https://doi.org/10.1111/j.1349-7006.2008.00779.x): SMAD4 sequence/deletion characterization; [ATCC mutation guide](https://delivery-files.atcc.org/api/public/content/255660-Cell-Lines-By-Gene-Mutation): Panc 03.27 and Panc 08.13 SMAD4 variants.
- [High-resolution pancreatic cancer genome analysis](https://pmc.ncbi.nlm.nih.gov/articles/PMC428474/): CDKN2A copy-number and expression evidence.
- Each row below links its specific Cellosaurus, ATCC, or primary-paper evidence; reported-WT calls have explicit supporting literature rather than being inferred from missing sequence-variation entries.

**Notation:** R₂ is the magnitude of the second circular moment, |mean(exp(2iθ))|. Figure 6I already uses this axial statistic on APA–mRNA phase differences; its corrected axis label is “Axial R₂”.


**Reconciliation with supplied `CCLE Genotyping Data.pdf` (3 pages).** The supplied report is a narrative reference summary without an identifiable DepMap release or underlying mutation/CN rows. Its proposed binary matrix is not adopted wholesale. Five display calls are revised using independent source evidence, including the subsequent Panc 10.05 literature refinement:

| Cell line / gene | Previous display | Revised display | Evidence and limitation |
| --- | --- | --- | --- |
| Hs 766T / CDKN2A | WT | mut | [NCI MutSpliceDB](https://brb.nci.nih.gov/cgi-bin/splicing/splicing_evidence.cgi?caid=CA373085801): NM_000077.4:c.457+2T>C with GDC/CCLE RNA-seq splice evidence; homozygosity unverified. |
| Panc 03.27 / CDKN2A | nd | del | [ATCC guide, p10 Table14](https://delivery-files.atcc.org/api/public/content/255660-Cell-Lines-By-Gene-Mutation): homozygous c.1_471del471; retain historical copy-number disagreement. |
| Panc 08.13 / CDKN2A | nd | del | Same ATCC table: homozygous c.1_471del471; reference-stock call. |
| Panc 08.13 / TP53 | WT | nd | [DepMap ACH-000417](https://depmap.org/portal/cell_line/ACH-000417) flags TP53_LoF, conflicting with reported WT; exact alteration unverified. |
| Panc 10.05 / CDKN2A | nd | WT | [2016 Panc10.05 study](https://pmc.ncbi.nlm.nih.gov/articles/PMC5097013/) explicitly reports p16(INK4A) WT; [Pa16C study](https://pmc.ncbi.nlm.nih.gov/articles/PMC10010283/) reports CDKN2A WT. |

Panc 03.27 SMAD4 remains **del**: ATCC p28 Table4 reports homozygous c.905_1659del755, contrary to the supplied report's no-clear-alteration summary. Panc 10.05 CDKN2A is **WT**, based on explicit p16/CDKN2A wild-type statements in the [2016 Panc10.05 study](https://pmc.ncbi.nlm.nih.gov/articles/PMC5097013/) and the [later Pa16C study](https://pmc.ncbi.nlm.nih.gov/articles/PMC10010283/). [Cellosaurus](https://www.cellosaurus.org/CVCL_1639) lists Pa16C as a synonym. SMAD4 remains **nd (conflicting reports)**: the 2016 study describes a Panc10.05 deletion, whereas the later study describes Pa16C as WT. These are reference annotations; synonymous names do not establish that different laboratories used identical stocks. Current DepMap bulk mutation and CN records were inaccessible during this check, so none of these calls is described as a newly validated current-release DepMap genotype. The source table preserves the report's claim, the previous call, the revised call, and independent evidence for all 40 entries. The supplied report's SHA-256 is `cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08`.


In [ ]:
# Cell 6: Audit driver annotations against literature and curated reference records
# Embedded evidence snapshot: no web access or external helper is needed to regenerate S16F.
DRIVER_EVIDENCE = [
    {'gene': 'KRAS', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.G12D (c.35G>A), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0152', 'pmid_or_doi': 'PMID:8026879;11115575;15367885', 'notes': 'Curated Cellosaurus/COSMIC/DepMap concordance.', 'cell': 'AsPC-1', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0152', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'TP53', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.C135Afs*35 (c.403delT), heterozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0152', 'pmid_or_doi': 'PMID:8026879;11169959;15367885', 'notes': 'Reference variant/deletion annotation; not genotyping of the exact BioCycle stocks.', 'cell': 'AsPC-1', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0152', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'SMAD4', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.R100T (c.299G>C), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0152', 'pmid_or_doi': 'PMID:10408907;18380791', 'notes': 'Reference variant/deletion annotation; not genotyping of the exact BioCycle stocks.', 'cell': 'AsPC-1', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0152', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'CDKN2A', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.L78Hfs*41 (c.233_234delTC), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0152', 'pmid_or_doi': 'PMID:11787853', 'notes': 'Reference variant/deletion annotation; not genotyping of the exact BioCycle stocks.', 'cell': 'AsPC-1', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0152', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'KRAS', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.G12D (c.35G>A), heterozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0480', 'pmid_or_doi': 'PMID:7809022;7961102;15367885', 'notes': 'Curated Cellosaurus.', 'cell': 'PANC-1', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0480', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'TP53', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.R273H (c.818G>A), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0480', 'pmid_or_doi': 'PMID:1764370;7961102;15367885', 'notes': 'Curated Cellosaurus.', 'cell': 'PANC-1', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0480', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'SMAD4', 'proposed_status': 'WT_reported', 'variant_or_interpretation': 'Reported wild type', 'evidence_url': 'https://aacrjournals.org/clincancerres/article/8/11/3628/289058/Suppression-of-Tumorigenesis-and-Induction-of', 'pmid_or_doi': 'Clinical Cancer Research 8:3628-3638 (2002)', 'notes': 'Reported WT in primary Smad4 transduction study; not genotyping of the present stocks.', 'cell': 'PANC-1', 'checked_on': '2026-09-23', 'display_status': 'WT', 'pre_report_display_status': 'WT', 'pre_report_evidence_url': 'https://aacrjournals.org/clincancerres/article/8/11/3628/289058/Suppression-of-Tumorigenesis-and-Induction-of', 'attached_report_call': 'WT', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'CDKN2A', 'proposed_status': 'hom_del', 'variant_or_interpretation': 'Homozygous deletion', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0480', 'pmid_or_doi': 'PMID:11787853;15367885', 'notes': 'Curated Cellosaurus.', 'cell': 'PANC-1', 'checked_on': '2026-09-23', 'display_status': 'del', 'pre_report_display_status': 'del', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0480', 'attached_report_call': 'del', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'KRAS', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.G12C (c.34G>T), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0428', 'pmid_or_doi': 'PMID:7961102;8026879;15367885', 'notes': 'Curated Cellosaurus.', 'cell': 'MIA PaCa-2', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0428', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'TP53', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.R248W (c.742C>T), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0428', 'pmid_or_doi': 'PMID:1764370;7961102;15367885', 'notes': 'Curated Cellosaurus.', 'cell': 'MIA PaCa-2', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0428', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'SMAD4', 'proposed_status': 'WT_reported', 'variant_or_interpretation': 'Reported wild type by coding-sequence analysis', 'evidence_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11158928/', 'pmid_or_doi': 'PMID:18380791;DOI:10.1111/j.1349-7006.2008.00779.x', 'notes': 'Primary study reports direct sequencing of coding exons and flanking intronic sequence.', 'cell': 'MIA PaCa-2', 'checked_on': '2026-09-23', 'display_status': 'WT', 'pre_report_display_status': 'WT', 'pre_report_evidence_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11158928/', 'attached_report_call': 'WT', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'CDKN2A', 'proposed_status': 'hom_del', 'variant_or_interpretation': 'Homozygous deletion', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0428', 'pmid_or_doi': 'PMID:11787853', 'notes': 'Curated Cellosaurus.', 'cell': 'MIA PaCa-2', 'checked_on': '2026-09-23', 'display_status': 'del', 'pre_report_display_status': 'del', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0428', 'attached_report_call': 'del', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'KRAS', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.G12V (c.35G>T), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0237', 'pmid_or_doi': 'PMID:8026879;8426738;15367885', 'notes': 'Curated Cellosaurus.', 'cell': 'Capan-1', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0237', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'TP53', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.A159V (c.476C>T), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0237', 'pmid_or_doi': 'PMID:8026879;8426738;15367885', 'notes': 'Curated Cellosaurus.', 'cell': 'Capan-1', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0237', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'SMAD4', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.S343* (c.1028C>G), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0237', 'pmid_or_doi': 'PMID:10408907;21750719', 'notes': 'Reference variant/deletion annotation; not genotyping of the exact BioCycle stocks.', 'cell': 'Capan-1', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0237', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'CDKN2A', 'proposed_status': 'hom_del', 'variant_or_interpretation': 'Homozygous deletion', 'evidence_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC428474/', 'pmid_or_doi': 'PMCID:PMC428474', 'notes': 'Primary array-CGH and expression study, Figure 3C, identifies Capan-1 among homozygously deleted lines.', 'cell': 'Capan-1', 'checked_on': '2026-09-23', 'display_status': 'del', 'pre_report_display_status': 'del', 'pre_report_evidence_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC428474/', 'attached_report_call': 'del', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'KRAS', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.G12V (c.35G>T), heterozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0026', 'pmid_or_doi': 'PMID:7961102;8026879;15367885', 'notes': 'Curated Cellosaurus.', 'cell': 'Capan-2', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0026', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'TP53', 'proposed_status': 'mut_splice', 'variant_or_interpretation': 'p.T125T (c.375G>T), homozygous; dramatically impairs splicing', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0026;https://pmc.ncbi.nlm.nih.gov/articles/PMC4451114/', 'pmid_or_doi': 'PMID:7961102;8026879;24700732', 'notes': 'Reference variant/deletion annotation; not genotyping of the exact BioCycle stocks.', 'cell': 'Capan-2', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0026;https://pmc.ncbi.nlm.nih.gov/articles/PMC4451114/', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'SMAD4', 'proposed_status': 'WT_reported', 'variant_or_interpretation': 'Reported wild type in comparison panel', 'evidence_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6135691/', 'pmid_or_doi': 'PMCID:PMC6135691', 'notes': 'Reference genotype in primary study comparison table; not genotyping of the present stocks.', 'cell': 'Capan-2', 'checked_on': '2026-09-23', 'display_status': 'WT', 'pre_report_display_status': 'WT', 'pre_report_evidence_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6135691/', 'attached_report_call': 'WT', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'CDKN2A', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.T18_A19dup (c.52_57dupACGGCC), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0026', 'pmid_or_doi': 'PMID:7972006', 'notes': 'Reference variant/deletion annotation; not genotyping of the exact BioCycle stocks.', 'cell': 'Capan-2', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0026', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'KRAS', 'proposed_status': 'WT_reported', 'variant_or_interpretation': 'Reported wild type', 'evidence_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9282935/', 'pmid_or_doi': 'PMCID:PMC9282935', 'notes': 'Explicit KRAS-WT reference line in primary study; agrees with historical genotyping, not inferred from an absent database row.', 'cell': 'BxPC-3', 'checked_on': '2026-09-23', 'display_status': 'WT', 'pre_report_display_status': 'WT', 'pre_report_evidence_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9282935/', 'attached_report_call': 'WT', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'TP53', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.Y220C (c.659A>G), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0186', 'pmid_or_doi': 'PMID:7961102;8026879;15367885', 'notes': 'Curated Cellosaurus.', 'cell': 'BxPC-3', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0186', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'SMAD4', 'proposed_status': 'hom_del', 'variant_or_interpretation': 'Homozygous deletion', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0186', 'pmid_or_doi': 'PMID:9788440;10408907;15367885', 'notes': 'Reference variant/deletion annotation; not genotyping of the exact BioCycle stocks.', 'cell': 'BxPC-3', 'checked_on': '2026-09-23', 'display_status': 'del', 'pre_report_display_status': 'del', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0186', 'attached_report_call': 'del', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'CDKN2A', 'proposed_status': 'hom_del', 'variant_or_interpretation': 'Homozygous deletion', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0186', 'pmid_or_doi': 'PMID:11169959', 'notes': 'Curated Cellosaurus.', 'cell': 'BxPC-3', 'checked_on': '2026-09-23', 'display_status': 'del', 'pre_report_display_status': 'del', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0186', 'attached_report_call': 'del', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'KRAS', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.Q61H (c.183A>C), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0334', 'pmid_or_doi': 'PMID:11169959', 'notes': 'Curated Cellosaurus; corrects historical false no-KRAS reports.', 'cell': 'Hs 766T', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0334', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'TP53', 'proposed_status': 'WT_reported', 'variant_or_interpretation': 'Reported wild type / no mutation reported', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0334;https://www.tandfonline.com/doi/full/10.1080/15384047.2024.2421584', 'pmid_or_doi': 'PMID:8026879;15367885;DOI:10.1080/15384047.2024.2421584', 'notes': 'Primary CF10 study identifies Hs 766T as exception to TP53-mutant lines. Cellosaurus says None_reported and flags an erroneous historical variant. Reported WT is not a new assay result.', 'cell': 'Hs 766T', 'checked_on': '2026-09-23', 'display_status': 'WT', 'pre_report_display_status': 'WT', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0334;https://www.tandfonline.com/doi/full/10.1080/15384047.2024.2421584', 'attached_report_call': 'WT', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'SMAD4', 'proposed_status': 'hom_del', 'variant_or_interpretation': 'Homozygous deletion', 'evidence_url': 'https://www.cellosaurus.org/CVCL_0334', 'pmid_or_doi': 'PMID:10408907;15367885;18380791', 'notes': 'Reference variant/deletion annotation; not genotyping of the exact BioCycle stocks.', 'cell': 'Hs 766T', 'checked_on': '2026-09-23', 'display_status': 'del', 'pre_report_display_status': 'del', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_0334', 'attached_report_call': 'del', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'CDKN2A', 'proposed_status': 'mut_splice', 'variant_or_interpretation': 'NM_000077.4:c.457+2T>C; abnormal CDKN2A splicing; zygosity unverified', 'evidence_url': 'https://brb.nci.nih.gov/cgi-bin/splicing/splicing_evidence.cgi?caid=CA373085801', 'pmid_or_doi': 'NCI MutSpliceDB allele CA373085801;GDC/CCLE Hs 766T;DepMap ACH-000178', 'notes': 'NCI GDC/CCLE RNA-seq evidence reports intron inclusion between exons 2 and 3 or skipping the end of exon 2 (hg19; G26233.Hs_766T.2.bam). This supersedes the prior reference-WT call. The accessible record does not establish homozygosity or a current DepMap release-level genotype.', 'cell': 'Hs 766T', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'WT', 'pre_report_evidence_url': 'https://www.tandfonline.com/doi/full/10.1080/15384047.2024.2421584', 'attached_report_call': 'mut?', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'KRAS', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.G12V (c.35G>T), heterozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_1635.txt', 'pmid_or_doi': 'ATCC:CRL-2549;COSMIC-CLP:925346;DepMap:ACH-000139', 'notes': 'Curated Cellosaurus.', 'cell': 'Panc 03.27', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_1635.txt', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'TP53', 'proposed_status': 'mut_splice', 'variant_or_interpretation': 'c.375+5G>T, homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_1635.txt', 'pmid_or_doi': 'COSMIC-CLP:925346', 'notes': 'Curated Cellosaurus.', 'cell': 'Panc 03.27', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_1635.txt', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'SMAD4', 'proposed_status': 'deletion', 'variant_or_interpretation': 'c.905_1659del755, reported ATCC/COSMIC annotation', 'evidence_url': 'https://delivery-files.atcc.org/api/public/content/255660-Cell-Lines-By-Gene-Mutation', 'pmid_or_doi': 'ATCC gene-mutation guide', 'notes': 'ATCC guide p28 Table4 explicitly reports homozygous c.905_1659del755. Retain the reported deletion despite the supplied summary stating no clear alteration; absence from a curated mutation list does not establish WT. Current DepMap release-level CN/mutation record was not inspected.', 'cell': 'Panc 03.27', 'checked_on': '2026-09-23', 'display_status': 'del', 'pre_report_display_status': 'del', 'pre_report_evidence_url': 'https://delivery-files.atcc.org/api/public/content/255660-Cell-Lines-By-Gene-Mutation', 'attached_report_call': 'no clear alteration', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'CDKN2A', 'proposed_status': 'hom_del_reported', 'variant_or_interpretation': 'Reported homozygous deletion c.1_471del471 (ATCC/COSMIC)', 'evidence_url': 'https://delivery-files.atcc.org/api/public/content/255660-Cell-Lines-By-Gene-Mutation;https://pmc.ncbi.nlm.nih.gov/articles/PMC428474/', 'pmid_or_doi': 'ATCC gene-mutation guide p10 Table14;PMCID:PMC428474', 'notes': 'ATCC explicitly records homozygous deletion. Older array-CGH Figure 3C reported retained locus/low expression, so retain the historical stock/platform discrepancy; reference call, not matched-stock genotyping.', 'cell': 'Panc 03.27', 'checked_on': '2026-09-23', 'display_status': 'del', 'pre_report_display_status': 'nd', 'pre_report_evidence_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC428474/;https://www.cellosaurus.org/CVCL_1635.txt', 'attached_report_call': 'del', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'KRAS', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.G12D (c.35G>A), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_1638.txt', 'pmid_or_doi': 'PMID:9612602;ATCC:CRL-2551', 'notes': 'Curated Cellosaurus.', 'cell': 'Panc 08.13', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_1638.txt', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'TP53', 'proposed_status': 'unresolved', 'variant_or_interpretation': 'DepMap TP53_LoF subtype; exact mutation/CN mechanism unresolved', 'evidence_url': 'https://depmap.org/portal/cell_line/ACH-000417;https://pmc.ncbi.nlm.nih.gov/articles/PMC12944281/', 'pmid_or_doi': 'DepMap ACH-000417 molecular subtype;PMCID:PMC12944281', 'notes': 'DepMap metadata flags TP53_LoF while published reference tables report WT. No exact mutation or CN event was accessible. Do not infer a genotype from the subtype label alone; change the unqualified WT display to nd.', 'cell': 'Panc 08.13', 'checked_on': '2026-09-23', 'display_status': 'nd', 'pre_report_display_status': 'WT', 'pre_report_evidence_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC12944281/', 'attached_report_call': 'WT?', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'SMAD4', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.C123Mfs*2 (c.366dupA/c.366_367insA), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_1638.txt;https://delivery-files.atcc.org/api/public/content/255660-Cell-Lines-By-Gene-Mutation', 'pmid_or_doi': 'ATCC:CRL-2551', 'notes': 'Reference variant/deletion annotation; not genotyping of the exact BioCycle stocks.', 'cell': 'Panc 08.13', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_1638.txt;https://delivery-files.atcc.org/api/public/content/255660-Cell-Lines-By-Gene-Mutation', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'CDKN2A', 'proposed_status': 'hom_del_reported', 'variant_or_interpretation': 'Reported homozygous deletion c.1_471del471 (ATCC/COSMIC)', 'evidence_url': 'https://delivery-files.atcc.org/api/public/content/255660-Cell-Lines-By-Gene-Mutation;https://pmc.ncbi.nlm.nih.gov/articles/PMC428474/', 'pmid_or_doi': 'ATCC gene-mutation guide p10 Table14;PMCID:PMC428474', 'notes': 'ATCC explicitly records homozygous deletion. Older array-CGH Figure 3C reported retained locus/low expression, so retain the historical stock/platform discrepancy; reference call, not matched-stock genotyping.', 'cell': 'Panc 08.13', 'checked_on': '2026-09-23', 'display_status': 'del', 'pre_report_display_status': 'nd', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_1638.txt', 'attached_report_call': 'del', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'KRAS', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.G12D (c.35G>A), heterozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_1639', 'pmid_or_doi': 'PMID:9612602;ATCC:CRL-2547;COSMIC-CLP:925348', 'notes': 'Curated Cellosaurus.', 'cell': 'Panc 10.05', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_1639', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'TP53', 'proposed_status': 'mut', 'variant_or_interpretation': 'p.I255N (c.764T>A), homozygous', 'evidence_url': 'https://www.cellosaurus.org/CVCL_1639', 'pmid_or_doi': 'COSMIC-CLP:925348;DepMap:ACH-000060', 'notes': 'Curated Cellosaurus; ATCC reports heterozygous, so retain Cellosaurus zygosity if needed.', 'cell': 'Panc 10.05', 'checked_on': '2026-09-23', 'display_status': 'mut', 'pre_report_display_status': 'mut', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_1639', 'attached_report_call': 'mut', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'SMAD4', 'proposed_status': 'conflicting_reports', 'variant_or_interpretation': 'Conflicting published SMAD4 deletion (Panc10.05) and WT (Pa16C) calls', 'evidence_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC5097013/;https://pmc.ncbi.nlm.nih.gov/articles/PMC10010283/;https://www.cellosaurus.org/CVCL_1639', 'pmid_or_doi': 'PMCID:PMC5097013;PMCID:PMC10010283;Cellosaurus CVCL_1639', 'notes': '2016 MCT study reports a Panc10.05 SMAD4 deletion; later VCP/p97 study describes Pa16C SMAD4 WT. Cellosaurus lists Pa16C as a synonym, but identical laboratory stocks are not established. Retain unresolved (nd) because the published reference calls conflict.', 'cell': 'Panc 10.05', 'checked_on': '2026-09-23', 'display_status': 'nd', 'pre_report_display_status': 'nd', 'pre_report_evidence_url': 'https://www.cellosaurus.org/CVCL_1639', 'attached_report_call': 'no reported alteration', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
    {'gene': 'CDKN2A', 'proposed_status': 'WT_reported', 'variant_or_interpretation': 'Explicitly reported p16/CDKN2A wild type (Panc10.05 and Pa16C)', 'evidence_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC5097013/;https://pmc.ncbi.nlm.nih.gov/articles/PMC10010283/;https://www.cellosaurus.org/CVCL_1639', 'pmid_or_doi': 'PMCID:PMC5097013;PMCID:PMC10010283;Cellosaurus CVCL_1639', 'notes': '2016 MCT study explicitly calls Panc10.05 p16(INK4A) WT; later VCP/p97 study calls Pa16C CDKN2A WT. Cellosaurus lists Pa16C as a synonym. WT is supported by explicit primary reports, not absence of a database alteration. Low expression does not establish a genomic alteration. Reference genotype, not matched-stock sequencing.', 'cell': 'Panc 10.05', 'checked_on': '2026-09-23', 'display_status': 'WT', 'pre_report_display_status': 'nd', 'pre_report_evidence_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC8260447/', 'attached_report_call': 'no reported alteration', 'report_filename': 'CCLE Genotyping Data.pdf', 'report_sha256': 'cae30c9c971cb6aea7d6cd8cf7784dd43af429cfa22e504304daee855d189d08'},
]
driver_evidence = pd.DataFrame(DRIVER_EVIDENCE)
cell_to_line = metrics.set_index('cell')['line'].to_dict()
driver_evidence['line'] = driver_evidence['cell'].map(cell_to_line)
assert driver_evidence['line'].notna().all()
assert len(driver_evidence) == 40 and not driver_evidence.duplicated(['line','gene']).any()
assert driver_evidence.groupby('line')['gene'].nunique().eq(4).all()
assert set(driver_evidence.display_status) <= {'WT','mut','del','nd'}
assert driver_evidence.evidence_url.str.startswith('https://').all()
drivers_upstream = drivers.copy(deep=True)
upstream_index = drivers_upstream.set_index('line')
driver_evidence['upstream_annotation'] = [
    upstream_index.loc[r.line, r.gene] for r in driver_evidence.itertuples()]
driver_calls = driver_evidence.pivot(index='line', columns='gene', values='display_status')
drivers = drivers.drop(columns=['KRAS','TP53','SMAD4','CDKN2A','src']).merge(
    driver_calls, left_on='line', right_index=True, validate='one_to_one')
drivers['src'] = 'Literature/curated-reference audit 2026-09-23; see driver evidence table'
write_source_table(drivers_upstream, 'fig6_s16_s17_driver_annotations_upstream.tsv')
write_source_table(driver_evidence, 'fig6_s16_s17_driver_evidence.tsv')
write_source_table(driver_evidence[['cell','gene','pre_report_display_status','attached_report_call',
    'display_status','variant_or_interpretation','evidence_url','notes','report_filename','report_sha256']],
    'fig6_s16_s17_ccle_report_reconciliation.tsv')
print('Audited 40 gene-by-cell-line reference annotations; nd means unresolved, not WT.')
display(driver_evidence.pivot(index='cell', columns='gene', values='display_status').reindex(
    [label_map[line] for line in label_order]))
display(driver_evidence[['cell','gene','upstream_annotation','display_status',
                         'variant_or_interpretation','evidence_url','notes']])


In [ ]:
# Cell 7: Export Figure 6/S16/S17 source tables and assemble plot-data contract
exports = {
    'fig6_s16_s17_metrics_current.tsv': metrics, 'fig6_s16_s17_supports_recomputed.tsv': supports.reset_index(),
    'fig6_s16_s17_rank_recomputed.tsv': rank, 'fig6_s16_s17_overall_current.tsv': overall,
    'fig6_s16_s17_best_per_aspect_recomputed.tsv': best_per_aspect, 'fig6_s16_s17_driver_annotations_current.tsv': drivers,
    'fig6_s16_s17_method_comparison_recomputed.tsv': method_comparison, 'fig6_s16_s17_phase_density_stats.tsv': phase_stats,
    'fig6_s16_s17_phase_density_curves.tsv': phase_density, 'fig6_s16_s17_phase_values.tsv': phase_values,
    'fig6_s16_s17_core_clock_matrix.tsv': core, 'fig6_s16_s17_module_timing.tsv': module_table,
}
for name, frame in exports.items(): write_source_table(frame, name)
for line, hm in heatmaps.items():
    slug=re.sub(r'[^A-Za-z0-9]+','_',line).strip('_')
    z=hm['z'].copy(); z.columns=[f'ZT{int(x)}' if float(x).is_integer() else f'ZT{x:g}' for x in z.columns]; z.insert(0,'gene',z.index)
    write_source_table(z, f'fig6_s16_s17_heatmap_{slug}_row_zscore.tsv')
    write_source_table(hm['metadata'], f'fig6_s16_s17_heatmap_{slug}_genes.tsv')

data = {'metrics':metrics, 'rank':rank, 'overall':overall, 'best_per_aspect':best_per_aspect, 'drivers':drivers, 'label_order':label_order, 'label_map':label_map, 'supports':supports, 'method_comparison':method_comparison, 'module_table':module_table, 'core':core, 'phase_stats':phase_stats, 'phase_density':phase_density, 'phase_values':phase_values, 'heatmaps':heatmaps}
assert set(data['label_order']) == set(metrics.line)
assert len(phase_stats) == 40 and len(core) == len(metrics)*len(CORE_CLOCK)
assert phase_density.groupby(['line','layer']).size().eq(361).all(), 'Expected closed-loop 360-grid KDE curves'
print(f'Wrote {len(exports) + 2*len(heatmaps)} source tables to {TABLE_DIR}')
print('Boundary: metrics and cohort alignment are current processed validation inputs; driver annotations use the embedded literature audit; all figure-facing support/ranking, method comparison, phase summaries/KDE, module/core-clock tables, and heatmaps were recomputed in this notebook.')

# Compact source QC: phase availability and representative clock-gene calls.
print('Phase availability by layer and retained line:')
display(phase_stats.pivot(index='layer', columns='cell', values='n'))
print('Rhythmic core-clock calls:')
display(core.loc[core.is_rhythmic, ['cell', 'gene', 'p_value', 'acrophase']])


### Kronos reporter traces (Figure 6A–C)

A–C show BMAL1 (blue) and PER2 (orange) in MIA PaCa-2, Hs 766T, and Panc 08.13, respectively, read directly from `data/kronos/Kronos.xlsx`. Workbook identifiers are `MiaPaca2`, `Hs766t`, and `Paca0813`. This is a separate experimental workbook, not a `process.smk` output. Its time values are interpreted as hours; synchronization metadata, signal units, biological-replicate identifiers, and prior signal processing are not specified in the file.

The analysis reproduces the settings of `3.8.0_kronos.ipynb` with all required code embedded here. Retain 0–60 h for MIA PaCa-2 and Hs 766T, and 24–84 h for Panc 08.13 (inclusive bounds). Display time is relative to each window start; retain the original slightly offset BMAL1/PER2 timestamps and do not interpolate endpoints. These are prespecified legacy windows, not windows optimized in this notebook.

Fit each retained raw-signal trace by ordinary least squares to `M + bc*cos(2πt/24) + bs*sin(2πt/24)`. The 24-hour period is fixed, not estimated. No additional detrending, smoothing or outlier removal is applied. Standardize observations and fitted curves with the retained trace's mean and population SD (`ddof=0`). Dashed horizontal lines mark the standardized MESOR; dotted vertical lines mark the first fitted peak within the observed window. Export raw and standardized observations, 1,000-point fitted curves, parameters, and missingness/window QC.

The summary retains the legacy nominal zero-amplitude F test and BH adjustment across six traces for reproducibility. Dense serial measurements are not biological replicates, and the strong residual autocorrelation violates the independent-error assumption. These probabilities do not establish replicate-level rhythmicity; no significance stars are shown in A–C. The fixed-period fits describe waveform timing and amplitude within the selected windows.


In [ ]:
# Cell 8: Kronos reporter time-course source analysis
from pathlib import Path
from typing import Any

from openpyxl import load_workbook
from scipy import stats

KRONOS_PERIOD_H = 24.0
KRONOS_WINDOWS = {
    'MiaPaca2': (0.0, 60.0),
    'Hs766t': (0.0, 60.0),
    'Paca0813': (24.0, 84.0),
}
KRONOS_EXPECTED_SHEETS = set(KRONOS_WINDOWS)
KRONOS_EXPECTED_GENES = {'BMAL1', 'PER2'}


def kronos_clean_header(value: Any) -> str:
    return str(value).strip().lower() if value is not None else ''


def kronos_as_finite_number(value: Any, *, sheet: str, excel_row: int,
                            column: int, field: str) -> float | None:
    if value is None or (isinstance(value, str) and not value.strip()):
        return None
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, float, np.number)):
        raise ValueError(f'Unexpected nonnumeric {field} in {sheet}!R{excel_row}C{column}: {value!r}')
    number = float(value)
    if not np.isfinite(number):
        raise ValueError(f'Non-finite {field} in {sheet}!R{excel_row}C{column}: {value!r}')
    return number


def kronos_find_header_row(rows: list[tuple[Any, ...]], sheet: str) -> int:
    for index, row in enumerate(rows):
        if any(kronos_clean_header(value) in {'timepoint', 'timepoints'} for value in row):
            return index
    raise ValueError(f'No timepoint header found in worksheet {sheet!r}')


def read_kronos(path: str | Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Read valid Kronos time/signal pairs and per-trace missingness QC."""
    workbook = load_workbook(filename=Path(path), read_only=True, data_only=True)
    observations: list[dict[str, Any]] = []
    qc_rows: list[dict[str, Any]] = []
    gene_names = {'bmal': 'BMAL1', 'bmal1': 'BMAL1', 'per2': 'PER2'}
    try:
        for worksheet in workbook.worksheets:
            rows = list(worksheet.iter_rows(values_only=True))
            header_index = kronos_find_header_row(rows, worksheet.title)
            header = rows[header_index]
            time_columns = [i for i, value in enumerate(header)
                            if kronos_clean_header(value) in {'timepoint', 'timepoints'}]
            if not time_columns:
                raise ValueError(f'No timepoint columns found in worksheet {worksheet.title!r}')
            trace_specs: list[tuple[int, int, str]] = []
            for time_position, time_column in enumerate(time_columns):
                next_time = time_columns[time_position + 1] if time_position + 1 < len(time_columns) else len(header)
                for signal_column in range(time_column + 1, next_time):
                    gene_header = kronos_clean_header(header[signal_column])
                    if gene_header not in gene_names:
                        raise ValueError(
                            f'Unexpected signal header in {worksheet.title}!R{header_index + 1}C{signal_column + 1}: '
                            f'{header[signal_column]!r}'
                        )
                    trace_specs.append((time_column, signal_column, gene_names[gene_header]))
            if not trace_specs:
                raise ValueError(f'No supported signal columns found in worksheet {worksheet.title!r}')
            for time_column, signal_column, gene in trace_specs:
                counts = {'n_sheet_rows': 0, 'n_valid': 0, 'n_missing_time': 0,
                          'n_missing_signal': 0, 'n_both_missing': 0}
                trace_records: list[dict[str, Any]] = []
                for row_index, row in enumerate(rows[header_index + 1:], start=header_index + 2):
                    time_value = row[time_column] if time_column < len(row) else None
                    signal_value = row[signal_column] if signal_column < len(row) else None
                    time_h = kronos_as_finite_number(time_value, sheet=worksheet.title,
                                                     excel_row=row_index, column=time_column + 1, field='time')
                    signal = kronos_as_finite_number(signal_value, sheet=worksheet.title,
                                                     excel_row=row_index, column=signal_column + 1, field='signal')
                    counts['n_sheet_rows'] += 1
                    if time_h is None and signal is None:
                        counts['n_both_missing'] += 1
                    elif time_h is None:
                        counts['n_missing_time'] += 1
                    elif signal is None:
                        counts['n_missing_signal'] += 1
                    else:
                        counts['n_valid'] += 1
                        trace_records.append({'cell_line': worksheet.title, 'gene': gene,
                                              'time_h': time_h, 'signal': signal,
                                              'excel_row': row_index, 'sheet': worksheet.title})
                trace_data = pd.DataFrame(trace_records)
                if not trace_data.empty and trace_data['time_h'].duplicated().any():
                    repeated = trace_data.loc[trace_data['time_h'].duplicated(keep=False), 'time_h'].tolist()
                    raise ValueError(f'Duplicate valid times for {worksheet.title} {gene}: {repeated}')
                observations.extend(trace_records)
                qc_rows.append({'cell_line': worksheet.title, 'gene': gene, 'sheet': worksheet.title,
                                'time_column': time_column + 1, 'signal_column': signal_column + 1, **counts})
    finally:
        workbook.close()
    columns = ['cell_line', 'gene', 'time_h', 'signal', 'excel_row', 'sheet']
    return pd.DataFrame(observations, columns=columns), pd.DataFrame(qc_rows)


def fit_single_trace(time: Any, signal: Any, period_h: float = 24) -> dict[str, float]:
    """Fit legacy M + bc*cos(2πt/T) + bs*sin(2πt/T) by OLS."""
    if not np.isfinite(period_h) or period_h <= 0:
        raise ValueError('period_h must be a positive finite number')
    time_array = np.asarray(time, dtype=float).reshape(-1)
    signal_array = np.asarray(signal, dtype=float).reshape(-1)
    if time_array.size != signal_array.size:
        raise ValueError('time and signal must have equal lengths')
    valid = np.isfinite(time_array) & np.isfinite(signal_array)
    time_array, signal_array = time_array[valid], signal_array[valid]
    n = int(signal_array.size)
    if n < 3:
        raise ValueError('At least three finite time/signal pairs are required for cosinor fitting')
    if np.unique(time_array).size != n:
        raise ValueError('Duplicate finite timepoints are not allowed within a cosinor trace')
    order = np.argsort(time_array)
    time_array, signal_array = time_array[order], signal_array[order]
    omega = 2 * np.pi / period_h
    design = np.column_stack((np.ones(n), np.cos(omega * time_array), np.sin(omega * time_array)))
    coefficients, _, rank, _ = np.linalg.lstsq(design, signal_array, rcond=None)
    if rank < 3:
        raise ValueError('Cosinor design is rank-deficient; supply timepoints spanning the harmonic')
    fitted = design @ coefficients
    residual = signal_array - fitted
    sse = float(np.sum(residual ** 2))
    sst = float(np.sum((signal_array - signal_array.mean()) ** 2))
    r_squared = float(1 - sse / sst) if sst > 0 else np.nan
    rmse = float(np.sqrt(sse / n))
    intercept, beta_cos, beta_sin = map(float, coefficients)
    amplitude = float(np.hypot(beta_cos, beta_sin))
    peak_phase_h = float((np.arctan2(beta_sin, beta_cos) * period_h / (2 * np.pi)) % period_h)
    df_residual = n - 3
    sse_null = float(np.sum((signal_array - signal_array.mean()) ** 2))
    near_constant = sse_null <= np.finfo(float).eps * max(float(np.sum(signal_array ** 2)), float(n))
    if near_constant:
        peak_phase_h = np.nan
        f_statistic = p_value = np.nan
    elif df_residual <= 0:
        f_statistic = p_value = np.nan
    elif sse <= np.finfo(float).eps * max(sse_null, 1.0):
        f_statistic, p_value = np.inf, 0.0
    else:
        f_statistic = max(0.0, ((sse_null - sse) / 2) / (sse / df_residual))
        p_value = float(stats.f.sf(f_statistic, 2, df_residual))
    residual_lag1 = (float(np.corrcoef(residual[:-1], residual[1:])[0, 1])
                     if n >= 3 and np.std(residual[:-1]) > 0 and np.std(residual[1:]) > 0 else np.nan)
    return {'n_observations': n, 'period_h': float(period_h), 'intercept': intercept,
            'beta_cos': beta_cos, 'beta_sin': beta_sin, 'amplitude': amplitude,
            'peak_phase_h': peak_phase_h, 'acrophase_h': peak_phase_h, 'mesor': intercept,
            'r_squared': r_squared, 'rmse': rmse, 'f_statistic': f_statistic,
            'p_value_nominal': p_value, 'df_numerator': 2, 'df_residual': df_residual,
            'residual_lag1_correlation': residual_lag1}


def kronos_bh_adjust(p_values: pd.Series) -> pd.Series:
    valid = p_values.notna()
    adjusted = pd.Series(np.nan, index=p_values.index, dtype=float)
    if valid.any():
        values = p_values.loc[valid].to_numpy(float)
        order = np.argsort(values)
        ranked = values[order] * len(values) / np.arange(1, len(values) + 1)
        ranked = np.minimum.accumulate(ranked[::-1])[::-1]
        result = np.empty_like(ranked)
        result[order] = np.minimum(ranked, 1.0)
        adjusted.loc[valid] = result
    return adjusted


def fit_cosinor(kronos_fit_data: pd.DataFrame, period_h: float = 24) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    required = {'cell_line', 'gene', 'time_h', 'signal', 'excel_row', 'sheet'}
    missing = required.difference(kronos_fit_data.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    summary_rows: list[dict[str, Any]] = []
    observed_parts: list[pd.DataFrame] = []
    curve_rows: list[dict[str, Any]] = []
    for (cell_line, gene), trace in kronos_fit_data.groupby(['cell_line', 'gene'], sort=True):
        trace = trace.copy().sort_values('time_h')
        result = fit_single_trace(trace['time_h'], trace['signal'], period_h=period_h)
        omega = 2 * np.pi / period_h
        fitted = result['intercept'] + result['beta_cos'] * np.cos(omega * trace['time_h'].to_numpy()) + result['beta_sin'] * np.sin(omega * trace['time_h'].to_numpy())
        trace['fitted'] = fitted
        trace['residual'] = trace['signal'].to_numpy() - fitted
        signal_mean = float(trace['signal'].mean())
        signal_sd = float(trace['signal'].std(ddof=0))
        trace['signal_zscore'] = ((trace['signal'] - signal_mean) / signal_sd if signal_sd > 0 else np.nan)
        trace['fitted_zscore'] = ((trace['fitted'] - signal_mean) / signal_sd if signal_sd > 0 else np.nan)
        observed_parts.append(trace)
        summary_rows.append({'cell_line': cell_line, 'gene': gene, 'sheet': trace['sheet'].iloc[0],
                             'signal_mean': signal_mean, 'signal_sd_ddof0': signal_sd, **result})
        grid = np.linspace(float(trace['time_h'].min()), float(trace['time_h'].max()), 1000)
        prediction = result['intercept'] + result['beta_cos'] * np.cos(omega * grid) + result['beta_sin'] * np.sin(omega * grid)
        for time_h, value in zip(grid, prediction):
            curve_rows.append({'cell_line': cell_line, 'gene': gene, 'sheet': trace['sheet'].iloc[0],
                               'time_h': float(time_h), 'fitted': float(value),
                               'fitted_zscore': float((value - signal_mean) / signal_sd) if signal_sd > 0 else np.nan})
    summary = pd.DataFrame(summary_rows)
    summary['p_value_bh'] = kronos_bh_adjust(summary['p_value_nominal'])
    observed = pd.concat(observed_parts, ignore_index=True) if observed_parts else kronos_fit_data.copy()
    return summary, observed, pd.DataFrame(curve_rows)


register_input(KRONOS_PATH)
kronos_all_data, kronos_qc = read_kronos(KRONOS_PATH)
assert set(kronos_all_data['cell_line']) == KRONOS_EXPECTED_SHEETS
assert set(kronos_all_data['gene']) == KRONOS_EXPECTED_GENES
assert kronos_all_data.groupby(['cell_line', 'gene']).ngroups == 6
assert not kronos_all_data.duplicated(['cell_line', 'gene', 'time_h']).any()

kronos_parts = []
for kronos_cell_line, (kronos_start_h, kronos_end_h) in KRONOS_WINDOWS.items():
    kronos_trace = kronos_all_data.loc[kronos_all_data['cell_line'].eq(kronos_cell_line)].copy()
    kronos_trace = kronos_trace.loc[kronos_trace['time_h'].between(kronos_start_h, kronos_end_h, inclusive='both')].copy()
    kronos_trace['plot_time_h'] = kronos_trace['time_h'] - kronos_start_h
    kronos_parts.append(kronos_trace)
kronos_data = pd.concat(kronos_parts, ignore_index=True)
windowQC = kronos_data.groupby(['cell_line', 'gene'], sort=False).agg(
    n_retained=('signal', 'size'), retained_first_h=('time_h', 'min'), retained_last_h=('time_h', 'max'),
    plot_first_h=('plot_time_h', 'min'), plot_last_h=('plot_time_h', 'max')).reset_index()
kronos_qc = kronos_qc.merge(windowQC, on=['cell_line', 'gene'], how='left', validate='one_to_one')
kronos_qc['n_excluded_by_window'] = kronos_qc['n_valid'] - kronos_qc['n_retained']
kronos_qc['window_start_h'] = kronos_qc['cell_line'].map(lambda value: KRONOS_WINDOWS[value][0])
kronos_qc['window_end_h'] = kronos_qc['cell_line'].map(lambda value: KRONOS_WINDOWS[value][1])

assert kronos_data.groupby(['cell_line', 'gene']).ngroups == 6
assert kronos_data.groupby(['cell_line', 'gene'])['time_h'].size().eq([241, 241, 241, 241, 240, 240]).all()
assert kronos_data.groupby(['cell_line', 'gene'])['time_h'].apply(lambda value: value.is_monotonic_increasing).all()
assert kronos_data.groupby(['cell_line', 'gene'])['plot_time_h'].apply(lambda value: value.between(0, 60).all()).all()
assert (kronos_qc['n_retained'] + kronos_qc['n_excluded_by_window']).eq(kronos_qc['n_valid']).all()

kronos_summary, kronos_observed, kronos_curves = fit_cosinor(kronos_data, period_h=KRONOS_PERIOD_H)
kronos_summary['window_start_h'] = kronos_summary['cell_line'].map(lambda value: KRONOS_WINDOWS[value][0])
kronos_summary['window_end_h'] = kronos_summary['cell_line'].map(lambda value: KRONOS_WINDOWS[value][1])
kronos_summary['display_time_offset_h'] = kronos_summary['window_start_h']
kronos_curves['plot_time_h'] = kronos_curves['time_h'] - kronos_curves['cell_line'].map(lambda value: KRONOS_WINDOWS[value][0])

assert len(kronos_summary) == 6 and len(kronos_observed) == len(kronos_data)
assert np.isfinite(kronos_observed[['signal', 'fitted', 'residual']]).all().all()
assert np.allclose(kronos_observed['signal'], kronos_observed['fitted'] + kronos_observed['residual'])
assert kronos_summary['period_h'].eq(KRONOS_PERIOD_H).all()
assert kronos_summary['acrophase_h'].between(0, KRONOS_PERIOD_H, inclusive='left').all()
assert kronos_curves.groupby(['cell_line', 'gene']).size().eq(1000).all()

for kronos_frame, kronos_name in [
    (kronos_qc, 'fig6_kronos_qc.tsv'),
    (kronos_all_data, 'fig6_kronos_input_long.tsv'),
    (kronos_data, 'fig6_kronos_fit_input_long.tsv'),
    (kronos_summary, 'fig6_kronos_cosinor_summary.tsv'),
    (kronos_observed, 'fig6_kronos_cosinor_observed.tsv'),
    (kronos_curves, 'fig6_kronos_cosinor_curves.tsv'),
]:
    write_source_table(kronos_frame, kronos_name)

display(kronos_qc[['cell_line', 'gene', 'n_valid', 'n_retained', 'n_excluded_by_window',
                    'retained_first_h', 'retained_last_h']])
display(kronos_summary[['cell_line', 'gene', 'n_observations', 'mesor', 'amplitude', 'acrophase_h',
                        'r_squared', 'p_value_nominal', 'p_value_bh', 'residual_lag1_correlation']])


## TCGA comparison added as S16A–B

These two panels reproduce the analysis from root notebook `3.9.0_cellline_tcga_representativeness.ipynb`, with all required analysis code embedded below and no runtime dependency on that notebook or its generated tables.

S16A projects the ten cell lines into PCA trained on 163 PDAC-compatible TCGA patients, using within-profile ranks across 9,467 shared expressed protein-coding genes and the 2,000 most variable genes selected from TCGA. S16B shows twelve core-clock gene TPM abundances as empirical percentiles in those patients. Cell-line points in A share their colors with column labels in B; B uses the existing S16 composite-rank order.

Use the common first-cycle phases ZT0, 4, 8, 12, 16 and 20 h, averaging replicate TPM within each phase and then weighting phases equally. ARNTL/MOP3 are normalized to BMAL1. Exact duplicated numeric columns at identical ZT are audited and removed. Processed BioCycle input rows are consistent with an upstream mean-TPM ≥1 screen; absent values are not set to zero. Select primary TCGA samples with ductal/adenocarcinoma morphology 8500/3 or 8140/3, excluding explicit neuroendocrine diagnoses. PCA is fit only to patient profiles; no batch correction is applied because assay and biological source are confounded.

These panels show transcriptomic proximity and clock-gene abundance, not proof of population representativeness or oscillator function. Patient bulk-tumor composition and STAR/GENCODE v36 versus cell-line Kallisto/GENCODE v43 quantification remain potential confounders. New inputs are the processed TCGA TPM matrix, GDC clinical annotations, GENCODE gene-type table and all ten expression time courses. Additional source/QC tables use the `s16_tcga_` prefix.

S16 now has five rows: **A(1), B(2), C(1)**; **D(1), E(1), F(2)**; **G(2), H(2)**; **I(4)**; **J(2), K(2)**, where numbers are widths in 52.5-mm square modules. Every row is one module high. The existing core-clock hit grid (I) is transposed to preserve readable 7-pt labels within that row, with no change to its values. PNGs omit panel letters; PDFs show A–K.


In [ ]:
# Cell 9: S16 TCGA-trained transcriptome PCA and core-clock abundance context
# Self-contained Figure S16 analysis fragment.  This deliberately reads primary
# processed inputs rather than legacy notebook outputs.
import csv
import hashlib
import re

import matplotlib as mpl
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA


def build_s16_tcga_expression(metrics_frame):
    """Rebuild the 3.9.0 TCGA PCA and core-clock percentile sources.

    Parameters
    ----------
    metrics_frame : pandas.DataFrame
        The current Figure 6 metrics table.  Its on-disk order supplies the
        ten-source cell-line order; it must contain ``line`` and ``cell``.

    Returns
    -------
    dict
        PCA scores/variance, 12-gene raw-TPM percentiles, the source cell
        order, optional colors, and auditable QC/source tables.
    """
    required_metrics = {'line', 'cell'}
    missing_metrics = required_metrics.difference(metrics_frame.columns)
    if missing_metrics:
        raise KeyError(f'S16 TCGA expression requires metrics columns: {sorted(missing_metrics)}')
    line_ids = metrics_frame['line'].astype(str).tolist()
    cell_order = metrics_frame['cell'].astype(str).tolist()
    if len(line_ids) != 10 or len(set(line_ids)) != 10 or len(set(cell_order)) != 10:
        raise AssertionError('Expected ten unique current Figure 6 cell lines in metrics order')
    line_to_cell = dict(zip(line_ids, cell_order))

    core_clock = ['BMAL1', 'CLOCK', 'NPAS2', 'CRY1', 'CRY2', 'PER1', 'PER2',
                  'PER3', 'NR1D1', 'NR1D2', 'DBP', 'TEF']
    aliases = {'ARNTL': 'BMAL1', 'MOP3': 'BMAL1'}
    primary_phases = (0., 4., 8., 12., 16., 20.)
    min_tpm, min_patient_fraction, n_variable_genes = 1., .20, 2000

    tcga_path = register_input(PROJECT_ROOT / 'data/Merged/Group_TPM_TCGA-PAAD_Tumor.csv')
    clinical_path = register_input(PROJECT_ROOT / 'data/reference/TCGA/clinical.tsv')
    gencode_path = register_input(PROJECT_ROOT / 'data/reference/gencode_gene_coordinates.tsv')

    def patient_id(sample_id):
        match = re.search(r'TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}', str(sample_id).strip().upper())
        if not match:
            raise ValueError(f'Not a TCGA barcode: {sample_id!r}')
        return match.group(0)

    def is_primary_01(sample_id):
        bits = str(sample_id).strip().upper().split('-')
        return len(bits) >= 4 and bits[3][:2] == '01'

    def clean_text(value):
        if pd.isna(value):
            return ''
        value = str(value).strip()
        return '' if value in {'', "'--", '--', 'NA', 'nan', 'None'} else value

    def clinical_primary_cohort(clinical, patient_ids):
        fields = ['cases.submitter_id', 'diagnoses.morphology', 'diagnoses.primary_diagnosis']
        absent = set(fields).difference(clinical.columns)
        if absent:
            raise KeyError(f'Clinical fields absent: {sorted(absent)}')
        clinical = clinical.loc[clinical['cases.submitter_id'].astype(str).str.fullmatch(
            r'TCGA-[A-Za-z0-9]{2}-[A-Za-z0-9]{4}')].copy()
        rows = []
        for participant, group in clinical.assign(
                patient_id=clinical['cases.submitter_id'].map(patient_id)).groupby('patient_id', sort=True):
            if participant not in patient_ids:
                continue
            morphologies = sorted({clean_text(x) for x in group['diagnoses.morphology'] if clean_text(x)})
            diagnoses = sorted({clean_text(x) for x in group['diagnoses.primary_diagnosis'] if clean_text(x)})
            diagnostic_text = ' '.join(morphologies + diagnoses).lower()
            ductal = any(code in morphologies for code in ('8500/3', '8140/3'))
            neuroendocrine = '8246/3' in morphologies or 'neuroendocrine' in diagnostic_text
            rows.append({
                'patient_id': participant,
                'primary_cohort': bool(ductal and not neuroendocrine),
                'morphology_all': ' | '.join(morphologies),
                'primary_diagnosis_all': ' | '.join(diagnoses),
                'pdac_8500_or_8140': ductal,
                'explicit_neuroendocrine': neuroendocrine,
                'selection_reason': ('exclude_explicit_neuroendocrine' if neuroendocrine else
                                     'include_ductal_or_adenocarcinoma_morphology' if ductal else
                                     'exclude_no_8500_3_or_8140_3'),
            })
        metadata = pd.DataFrame(rows).set_index('patient_id').reindex(sorted(patient_ids))
        metadata['primary_cohort'] = metadata['primary_cohort'].fillna(False).astype(bool)
        metadata['selection_reason'] = metadata['selection_reason'].fillna('exclude_missing_clinical_patient')
        return metadata

    # Full processed TCGA TPM matrix and the exact primary163 clinical rule.
    tcga = pd.read_csv(tcga_path, sep='\t', index_col=0)
    tcga.index = tcga.index.astype(str).str.strip().str.upper().map(lambda gene: aliases.get(gene, gene))
    tcga = tcga.apply(pd.to_numeric, errors='raise')
    if tcga.isna().any().any() or (tcga < 0).any().any():
        raise ValueError('TCGA TPM contains missing or negative values')
    tcga = tcga.groupby(level=0, sort=True).sum()
    sample_metadata = pd.DataFrame({'sample_id': tcga.columns.astype(str)})
    sample_metadata['patient_id'] = sample_metadata['sample_id'].map(patient_id)
    sample_metadata['is_primary_01'] = sample_metadata['sample_id'].map(is_primary_01)
    if not sample_metadata['is_primary_01'].all() or sample_metadata['patient_id'].duplicated().any():
        raise AssertionError('TCGA TPM must have one unique primary-01 sample per patient')
    if len(sample_metadata) != 178:
        raise AssertionError(f'Expected current 178 TCGA primary samples, found {len(sample_metadata)}')
    clinical = pd.read_csv(clinical_path, sep='\t', low_memory=False)
    patient_metadata = sample_metadata.set_index('patient_id').join(
        clinical_primary_cohort(clinical, set(sample_metadata['patient_id'])), how='left')
    patient_metadata['primary_cohort'] = patient_metadata['primary_cohort'].fillna(False).astype(bool)
    patients = patient_metadata.loc[patient_metadata['primary_cohort']].copy()
    if len(patients) != 163:
        raise AssertionError(f'Expected primary163 cohort after clinical selection, found {len(patients)}')
    tcga.columns = sample_metadata['patient_id'].tolist()
    tcga = tcga.loc[:, patients.index]

    def stable_hash(values):
        return hashlib.sha256(np.asarray(values.to_numpy(float), dtype='<f8').tobytes()).hexdigest()

    def load_primary_cell_tpm(line):
        path = register_input(CMO_DATA / 'biocycle_analysis/inputs' / f'input_{line}.txt')
        with path.open(newline='') as handle:
            header = next(csv.reader(handle, delimiter='\t'))
        raw = pd.read_csv(path, sep='\t', header=None, skiprows=1, dtype=str)
        raw.columns = ['symbol'] + header[1:]
        source_columns = raw.columns[1:]
        phases = []
        for column in source_columns:
            match = re.fullmatch(r'ZT(\d+(?:\.\d+)?)', str(column))
            if not match:
                raise ValueError(f'Unexpected BioCycle sample header {column!r} in {line}')
            phases.append(float(match.group(1)))
        values = raw.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')
        if values.isna().any().any() or not np.isfinite(values.to_numpy(float)).all():
            raise ValueError(f'Non-finite logged TPM in {line}')
        kept_positions, duplicate_rows, seen = [], [], {}
        for position, (phase, column) in enumerate(zip(phases, source_columns)):
            digest = stable_hash(values.iloc[:, position])
            key = (phase, digest)
            duplicate_of = seen.get(key)
            retained = duplicate_of is None
            if retained:
                seen[key] = position
                kept_positions.append(position)
            duplicate_rows.append({'line': line, 'source_column_position': position + 2,
                                   'source_header': str(column), 'zt_hour': phase,
                                   'column_hash': digest, 'retained': retained,
                                   'duplicate_of_position': np.nan if retained else duplicate_of + 2})
        recovered_tpm = np.expm1(values.iloc[:, kept_positions].to_numpy(float) * np.log(2.0))
        if not np.isfinite(recovered_tpm).all() or (recovered_tpm < -1e-10).any():
            raise ValueError(f'Invalid recovered TPM in {line}')
        symbols = raw['symbol'].astype(str).str.strip().str.upper().replace(aliases)
        tpm = pd.DataFrame(np.maximum(recovered_tpm, 0.), index=symbols,
                           columns=np.asarray([phases[i] for i in kept_positions], dtype=float))
        tpm = tpm.groupby(level=0, sort=True).sum()
        by_phase = tpm.T.groupby(level=0).mean().T
        if not set(primary_phases).issubset(by_phase.columns):
            raise AssertionError(f'{line} lacks one of primary ZT {primary_phases}: {list(by_phase.columns)}')
        profile = by_phase.loc[:, list(primary_phases)].mean(axis=1)
        qc = {'line': line, 'cell': line_to_cell[line], 'n_source_genes': len(raw),
              'n_canonical_genes': tpm.shape[0], 'n_source_columns': len(source_columns),
              'n_retained_columns': len(kept_positions),
              'n_exact_duplicate_columns_removed': len(source_columns) - len(kept_positions),
              'numeric_zt': ','.join(f'{phase:g}' for phase in by_phase.columns),
              'n_primary_phases': len(primary_phases),
              'n_core_clock_present': sum(gene in tpm.index for gene in core_clock)}
        return profile, pd.DataFrame(duplicate_rows), qc

    cell_profiles, duplicate_audits, line_qc = {}, [], []
    for line in line_ids:
        profile, duplicate_audit, qc = load_primary_cell_tpm(line)
        cell_profiles[line_to_cell[line]] = profile
        duplicate_audits.append(duplicate_audit)
        line_qc.append(qc)
    cell_tpm = pd.concat(cell_profiles, axis=1).reindex(columns=cell_order)
    cell_tpm.columns.name = 'cell'
    cell_values = cell_tpm.to_numpy(float)
    if (cell_tpm < 0).any().any() or not np.isfinite(cell_values[~np.isnan(cell_values)]).all():
        raise AssertionError('Cell-line primary TPM profiles must be finite and non-negative')

    # Exact feature universe, within-profile rank, and TCGA-trained scaled PCA.
    gene_reference = pd.read_csv(gencode_path, sep='\t')
    gene_reference['gene_name'] = gene_reference['gene_name'].replace({'ARNTL': 'BMAL1'})
    protein_coding = set(gene_reference.loc[gene_reference['gene_type'].eq('protein_coding'), 'gene_name'])
    shared = sorted(set(tcga.index) & set(cell_tpm.index) & protein_coding)
    measured = cell_tpm.loc[shared].notna().all(axis=1) & tcga.loc[shared].notna().all(axis=1)
    shared = list(measured.index[measured])
    expressed = tcga.loc[shared].ge(min_tpm).mean(axis=1).ge(min_patient_fraction)
    common_genes = list(expressed.index[expressed])
    if len(common_genes) < 3000:
        raise AssertionError('Too few shared expressed protein-coding genes; inspect mapping/scale')
    tcga_log = np.log2(tcga.loc[common_genes] + 1.)
    cell_log = np.log2(cell_tpm.loc[common_genes, cell_order] + 1.)
    patient_rank = tcga_log.rank(axis=0, method='average', pct=True)
    cell_rank = cell_log.rank(axis=0, method='average', pct=True)
    feature_iqr = (tcga_log.quantile(.75, axis=1) - tcga_log.quantile(.25, axis=1)).sort_values(
        ascending=False, kind='stable')
    variable_genes = feature_iqr.head(min(n_variable_genes, len(feature_iqr))).index.tolist()
    patient_matrix = patient_rank.loc[variable_genes].T
    means = patient_matrix.mean(axis=0)
    scales = patient_matrix.std(axis=0, ddof=0)
    pca_genes = scales.index[scales.gt(1e-8)].tolist()
    tcga_scaled = (patient_matrix.loc[:, pca_genes] - means[pca_genes]) / scales[pca_genes]
    cell_scaled = (cell_rank.loc[pca_genes].T - means[pca_genes]) / scales[pca_genes]
    pca = PCA(n_components=min(10, len(tcga_scaled) - 1), svd_solver='full')
    patient_pc = pca.fit_transform(tcga_scaled)
    cell_pc = pca.transform(cell_scaled)
    pc_columns = [f'PC{i + 1}' for i in range(patient_pc.shape[1])]
    pca_scores = pd.concat([
        pd.DataFrame(patient_pc, index=patient_matrix.index, columns=pc_columns).assign(source='TCGA', cell=''),
        pd.DataFrame(cell_pc, index=cell_order, columns=pc_columns).assign(source='Cell line', cell=cell_order),
    ])
    pca_scores.index.name = 'sample_id'
    pca_variance = pd.DataFrame({'component': np.arange(1, len(pca.explained_variance_ratio_) + 1),
                                 'explained_variance_fraction': pca.explained_variance_ratio_})
    pca_loadings = pd.DataFrame(pca.components_.T, index=pca_genes, columns=pc_columns)
    pca_loadings.index.name = 'gene'

    # Raw TPM percentiles exactly match the original 12-gene abundance panel.
    clock_percentiles = pd.DataFrame(index=core_clock, columns=cell_order, dtype=float)
    clock_rows, clock_coverage = [], []
    for gene in core_clock:
        present_tcga, present_cell = gene in tcga.index, gene in cell_tpm.index
        clock_coverage.append({'gene': gene, 'in_patient_matrix': present_tcga,
                               'in_cellline_matrix': present_cell,
                               'in_expressed_common_universe': gene in common_genes})
        if not (present_tcga and present_cell):
            continue
        reference = tcga.loc[gene].dropna().to_numpy(float)
        p025, median, p975 = np.quantile(reference, [.025, .5, .975])
        for cell in cell_order:
            value = float(cell_tpm.at[gene, cell])
            percentile = 100 * (np.count_nonzero(reference < value) +
                                .5 * np.count_nonzero(reference == value)) / len(reference)
            clock_percentiles.at[gene, cell] = percentile
            clock_rows.append({'gene': gene, 'cell': cell, 'cellline_tpm': value,
                               'cellline_log2_tpm': np.log2(value + 1.),
                               'tcga_tpm_median': median, 'tcga_tpm_p025': p025,
                               'tcga_tpm_p975': p975, 'tcga_percentile': percentile,
                               'within_patient_95pct_range': bool(p025 <= value <= p975)})
    clock_long = pd.DataFrame(clock_rows)
    if clock_percentiles.shape != (12, 10) or clock_percentiles.isna().any().any():
        raise AssertionError('Expected complete 12-clock-gene by 10-cell-line percentile matrix')
    clock_summary = clock_long.groupby('cell', sort=False).agg(
        n_clock_genes=('gene', 'nunique'),
        n_within_patient_range=('within_patient_95pct_range', 'sum')).reset_index()
    clock_summary['fraction_within_patient_range'] = (
        clock_summary['n_within_patient_range'] / clock_summary['n_clock_genes'])

    gene_universe = pd.DataFrame({
        'gene': common_genes,
        'patient_expressed_fraction': tcga.loc[common_genes].ge(min_tpm).mean(axis=1).to_numpy(),
        'selected_variable': [gene in set(variable_genes) for gene in common_genes],
    })
    qc = pd.DataFrame([
        {'stage': 'TCGA primary-01 samples', 'n': len(sample_metadata)},
        {'stage': 'TCGA primary163 clinical cohort', 'n': len(patients)},
        {'stage': 'Cell lines', 'n': len(cell_order)},
        {'stage': 'Shared measured protein-coding genes', 'n': len(shared)},
        {'stage': 'Patient-expressed shared genes', 'n': len(common_genes)},
        {'stage': 'TCGA-selected variable genes', 'n': len(variable_genes)},
        {'stage': 'Nonconstant PCA genes', 'n': len(pca_genes)},
    ])
    qc['primary_cohort_rule'] = '8500/3 or 8140/3 morphology; exclude explicit neuroendocrine morphology/diagnosis'
    qc['cell_profile_rule'] = 'recover linear TPM; mean within numeric ZT; equal mean across ZT0,4,8,12,16,20'
    qc['feature_rule'] = 'shared measured protein-coding; TCGA TPM >=1 in >=20%; top 2000 TCGA log2(TPM+1) IQR'
    qc['pca_rule'] = 'within-profile average rank; scale with TCGA means/SD (ddof=0); full-SVD PCA trained in TCGA'

    # Source-table and provenance contract for later S16 plotting/validation cells.
    write_source_table(pca_scores.reset_index(), 's16_tcga_pca_scores.tsv')
    write_source_table(pca_variance, 's16_tcga_pca_variance.tsv')
    write_source_table(pca_loadings.reset_index(), 's16_tcga_pca_loadings.tsv')
    write_source_table(clock_percentiles.rename_axis('gene').reset_index(), 's16_tcga_clock_percentiles.tsv')
    write_source_table(clock_long, 's16_tcga_clock_expression_comparison.tsv')
    write_source_table(pd.DataFrame(clock_coverage), 's16_tcga_clock_gene_mapping_qc.tsv')
    write_source_table(clock_summary, 's16_tcga_clock_coverage_by_line.tsv')
    write_source_table(gene_universe, 's16_tcga_gene_universe.tsv')
    write_source_table(pd.DataFrame(line_qc), 's16_tcga_cell_input_qc.tsv')
    write_source_table(pd.concat(duplicate_audits, ignore_index=True), 's16_tcga_duplicate_zt_audit.tsv')
    write_source_table(patient_metadata.reset_index(), 's16_tcga_patient_selection_audit.tsv')
    write_source_table(qc, 's16_tcga_qc.tsv')
    input_paths = [tcga_path, clinical_path, gencode_path] + [
        CMO_DATA / 'biocycle_analysis/inputs' / f'input_{line}.txt' for line in line_ids]
    provenance = pd.DataFrame([
        {'registered': Path(path) in INPUT_MANIFEST,
         **({'input_root': INPUT_MANIFEST[Path(path)].get('input_root', ''),
             'input_path': INPUT_MANIFEST[Path(path)].get('input_path', INPUT_MANIFEST[Path(path)].get('path', '')),
             'bytes': INPUT_MANIFEST[Path(path)].get('bytes', np.nan),
             'sha256': INPUT_MANIFEST[Path(path)].get('sha256', '')}
            if Path(path) in INPUT_MANIFEST else {})}
        for path in input_paths
    ])
    write_source_table(provenance, 's16_tcga_input_provenance.tsv')

    return {'pca_scores': pca_scores, 'pca_variance': pca.explained_variance_ratio_.copy(),
            'clock_percentiles': clock_percentiles, 'cell_order': cell_order,
            'colors': {cell: mpl.colormaps['tab10'](i) for i, cell in enumerate(cell_order)},
            'QC': {'summary': qc, 'line_inputs': pd.DataFrame(line_qc),
                   'patient_selection': patient_metadata, 'clock_coverage': pd.DataFrame(clock_coverage),
                   'clock_summary': clock_summary, 'gene_universe': gene_universe,
                   'duplicate_zt_audit': pd.concat(duplicate_audits, ignore_index=True),
                   'input_provenance': provenance, 'pca_variance_table': pca_variance}}


S16_TCGA = build_s16_tcga_expression(metrics)


## 3. Publication figures

Figure 6A–C display the Kronos reporter traces and fixed-period cosinor fits described above, with one shared legend in the assembly. Each is one 52.5 mm square module. Figure 6D summarizes retained-line support for the five displayed cohort axes. E shows the highest-ranked line's phase-sorted expression heatmap; F overlays normalized phase densities. G–J show current effect estimates, with color marking the existing nominal *p* < 0.05 result. K reports the fraction of measured genes in each curated module called rhythmic; L compares the composite alignment score with rhythmic expression burden.

S16A–B show the TCGA-trained transcriptome PCA and core-clock abundance percentiles. S16C–K give rankings, the retained-line harmonic/BioCycle comparison, best-per-axis examples, literature-audited driver annotations, source effect values, phase availability, core-clock calls and the next two ranked expression heatmaps. The effect-size heatmap is standardized per metric across cell lines; annotations retain the unscaled values (two significant figures; leading zeros omitted for fractions to prevent crowded text).

S17A shows four phase layers in the current cell-line rank order. The two five-line blocks belong to the same panel. The circular curves use a von Mises kernel (bandwidth 0.35 radians) on a closed 360-step grid, with paired dashed lines at the second-moment axis and its 12 h opposite. Cells with fewer than three phases are identified explicitly. S17B–E retain the four count/amplitude comparisons. These expanded grids use additional whole-module height to keep all text readable.
**Statistical notation.** Use lowercase italic *p* for p-values and lowercase italic *q* for FDR-adjusted values. The current Figure 6/S16–S17 panels contain no numeric p- or q-value annotations. Figure 6G–J encode nominal *p* < 0.05 by point color. Numerical statistical results remain in the source tables.


In [ ]:
# Cell 10: Figure 6–S17 plotting helpers
# All figures use whole 52.5-mm modules; save_figure supplies PDF-only panel tags.

PALETTE = {'blue': '#0072B2', 'orange': '#D55E00', 'green': '#009E73', 'pink': '#CC79A7', 'yellow': '#E69F00', 'cyan': '#56B4E9', 'gray': '#999999'}
LAYERS = ['Expression', 'Skipped exon', 'APA', 'RBP expression']
LAYER_COLORS = dict(zip(LAYERS, [PALETTE['blue'], PALETTE['orange'], PALETTE['green'], PALETTE['pink']]))
MAIN_MODULES = ['Core clock', 'Cell cycle', 'RAS/MAPK signaling', 'RNA processing/RBP', 'APA machinery', 'Splicing machinery']
CLAIM_SHORT = {
    'Core oscillator retained': 'Core oscillator',
    'Downstream transcription rewired/polarized': 'Expression',
    'Clock-coupling altered': 'Clock TF',
    'Rhythmic AS phase organized': 'Splicing',
    'Rhythmic APA coordinated with host timing': 'APA timing',
    'RBP/CLIP processing modules': 'RBP/CLIP',
    'Clock-TF coupling (OR)': 'Clock TF',
    'Rhythmic-gene polarization': 'Expr. polarization',
    'Rhythmic-SE polarization': 'Splicing polarization',
    'APA rhythmic amplitude': 'APA amplitude',
    'APA-mRNA lag concentration': 'APA–mRNA lag',
    'RBP acrophase organization': 'RBP phase',
    'CLIP-target APA enrichment (#RBP)': 'CLIP–APA RBPs',
    'Core-clock genes rhythmic (#)': 'Core-clock hits',
}

def _fig(w, h):
    return plt.figure(figsize=(w * PANEL_MM / 25.4, h * PANEL_MM / 25.4))

def _style(ax, grid='x'):
    ax.tick_params(labelsize=FONT['tick'], length=2.5, width=0.6)
    if grid:
        ax.grid(axis=grid, color='#E6E6E6', linewidth=0.55, zorder=0)
    ax.spines[['top', 'right']].set_visible(False)

def _blank(ax, message='No data available'):
    ax.set_axis_off()
    ax.text(.5, .5, message, transform=ax.transAxes, ha='center', va='center', fontsize=FONT['annotation'])

def _labels(frame, lines):
    return [str(frame.get('plot_label', frame.get('cell', frame.get('line'))).loc[l]) if hasattr(frame.get('plot_label', frame.get('cell', frame.get('line'))), 'loc') else str(l) for l in lines]

def _heatmap(ax, info, title, colorbar=True):
    if not info or getattr(info.get('z'), 'empty', True):
        _blank(ax, 'No rhythmic-gene heatmap')
        return
    z, times = info['z'], np.asarray(info['times'], dtype=float)
    cmap = mpl.colors.LinearSegmentedColormap.from_list('expr_z', ['#2166AC', '#FFFFFF', '#B2182B'])
    im = ax.imshow(z.values, aspect='auto', cmap=cmap, vmin=-2.5, vmax=2.5, interpolation='nearest', rasterized=True)
    ax.set_yticks([])
    ax.set_ylabel(f'{len(z):,} rhythmic genes\nsorted by BioCycle lag', fontsize=FONT['axis_title'])
    ax.set_xticks(range(len(times)))
    ax.set_xticklabels([f'ZT{x:g}' for x in times], fontsize=FONT['tick'])
    ax.set_title(f'{title}: {info.get("cell", info.get("line", ""))}\nrow-scaled rhythmic expression', fontsize=FONT['title'], pad=5)
    strip = ax.inset_axes([0, 1.02, 1, .045])
    strip.imshow((times % 24)[None, :], aspect='auto', cmap='twilight', vmin=0, vmax=24)
    strip.set_axis_off()
    if colorbar:
        cb = ax.figure.colorbar(im, ax=ax, fraction=.032, pad=.018)
        cb.set_label('Row z-score', fontsize=FONT['axis_title'], labelpad=2)
        cb.ax.tick_params(labelsize=FONT['tick'])

def _overall(ax):
    df = data['overall'].query("axis != 'Core oscillator retained'").copy().sort_values('support_fraction')
    df['short'] = df['axis'].map(CLAIM_SHORT).fillna(df['axis'])
    color = np.where(df.support_fraction >= .75, PALETTE['green'], np.where(df.support_fraction >= .5, PALETTE['yellow'], PALETTE['orange']))
    ax.barh(df.short, df.support_fraction, color=color, zorder=2)
    for y, row in enumerate(df.itertuples()):
        ax.text(row.support_fraction + .025, y, f'{int(row.support_lines)}/{int(row.n_lines)}', va='center', fontsize=FONT['annotation'])
    ax.set(xlim=(0, 1.36), xlabel='Fraction of lines', title='Cohort alignment')
    _style(ax)

def _lollipop(ax, value, title, xlabel, pcol=None):
    df = data['metrics'].sort_values(value).copy()
    values = pd.to_numeric(df[value], errors='coerce')
    colors = np.where(pd.to_numeric(df[pcol], errors='coerce') < .05, PALETTE['blue'], '#BDBDBD') if pcol else PALETTE['blue']
    ax.hlines(df.plot_label, 0, values, color='#CFCFCF', linewidth=.8, zorder=1)
    ax.scatter(values, df.plot_label, c=colors, s=24, zorder=2, edgecolors='none')
    ax.set(xlabel=xlabel, title=title)
    _style(ax)

def _combined_density(ax, line):
    have = False
    for layer in LAYERS:
        sub = phase_density.query('line == @line and layer == @layer').sort_values('theta')
        st = phase_stats.query('line == @line and layer == @layer')
        if sub.empty or st.empty or int(st.iloc[0]['n']) < 3:
            continue
        have = True
        row = st.iloc[0]
        theta, density = sub.theta.to_numpy(), sub.density.to_numpy()
        if theta[-1] < 2 * np.pi:
            theta, density = np.r_[theta, theta[0] + 2*np.pi], np.r_[density, density[0]]
        color = LAYER_COLORS[layer]
        ax.plot(theta, density, color=color, lw=1.1, label=f'{dict(zip(LAYERS,["Expr","SE","APA","RBP"]))[layer]} n={int(row.n)}; R₂={row.R2:.2f}')
        for angle in (row.axis_phase_rad, row.opposite_axis_phase_rad):
            ax.axvline(angle, color=color, lw=.65, ls='--', alpha=.7)
    ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
    ax.set_xticks(np.linspace(0, 2*np.pi, 6, endpoint=False)); ax.set_xticklabels(['0', '4', '8', '12', '16', '20'], fontsize=FONT['tick'])
    ax.set_yticklabels([])
    label = data['label_map'].get(line, line)
    ax.set_title('Phase density\n(dashed: axial orientation)', fontsize=FONT['title'], pad=10)
    if have:
        ax.legend(loc='lower center', bbox_to_anchor=(.5, .01), fontsize=FONT['legend'], ncol=1, handlelength=1.5, labelspacing=.25)

def _module_heatmap(ax):
    pivot = module_table.pivot(index='module', columns='line', values='frac_rhythmic').reindex(MAIN_MODULES).reindex(columns=data['label_order'])
    vmax = max(.05, np.nanmax(pivot.values))
    im = ax.imshow(pivot, aspect='auto', cmap='viridis', vmin=0, vmax=vmax)
    ax.set_xticks(range(pivot.shape[1])); ax.set_xticklabels([data['label_map'][x] for x in pivot.columns], rotation=45, ha='right', fontsize=FONT['tick'])
    ax.set_yticks(range(pivot.shape[0])); ax.set_yticklabels(['Core clock', 'Cell cycle', 'RAS/MAPK', 'RNA processing', 'APA machinery', 'Splicing machinery'], fontsize=FONT['tick'])
    ax.set_title('Rhythmic fraction in curated modules', fontsize=FONT['title'])
    cb = ax.figure.colorbar(im, ax=ax, fraction=.035, pad=.02); cb.set_label('Fraction rhythmic', fontsize=FONT['axis_title']); cb.ax.tick_params(labelsize=FONT['tick'])

def _summary(ax):
    cols = ['line', 'n_rhythmic_genes']
    df = data['rank'].merge(data['metrics'][cols], on='line', how='left')
    bins = pd.cut(df.composite, [-np.inf, 5.5, 6.5, np.inf], labels=['Lower', 'Intermediate', 'Higher'])
    cmap = {'Higher': PALETTE['green'], 'Intermediate': PALETTE['yellow'], 'Lower': PALETTE['orange']}
    for group in ['Higher', 'Intermediate', 'Lower']:
        sub = df[bins == group]; ax.scatter(sub.n_rhythmic_genes, sub.composite, s=28, label=group, color=cmap[group])
    for _, row in df.loc[df.composite.isin([df.composite.min(), df.composite.max()])].iterrows():
        ax.annotate(data['label_map'].get(row.line, row.line), (row.n_rhythmic_genes, row.composite), xytext=(2, 3), textcoords='offset points', fontsize=FONT['annotation'])
    ax.set(xlabel='Rhythmic expression genes', ylabel='Composite score', title='Cell-line alignment phenotypes')
    ax.legend(title='Alignment', fontsize=FONT['legend'], title_fontsize=FONT['legend'], loc='lower right')
    _style(ax)

def _rank(ax):
    df = data['rank'].sort_values('composite')
    c = np.where(df.composite.rank(pct=True) > .67, PALETTE['green'], np.where(df.composite.rank(pct=True) > .34, PALETTE['yellow'], PALETTE['orange']))
    ax.barh(df.plot_label, df.composite, color=c, zorder=2)
    ax.set(xlabel='Composite score', title='Composite alignment'); _style(ax)

def _method(ax):
    claims = ['Rhythmic-gene polarization', 'Clock-TF coupling (OR)', 'Rhythmic-SE polarization', 'APA-mRNA lag concentration', 'RBP acrophase organization']
    df = method_comparison[method_comparison.claim.isin(claims)].copy()
    y = np.arange(len(df))
    ax.scatter(df.support_fraction_harmonic, y+.13, color=PALETTE['gray'], s=24, label='Harmonic')
    ax.scatter(df.support_fraction_biocycle, y-.13, color=PALETTE['blue'], s=24, label='BioCycle')
    ax.set(yticks=y, yticklabels=[CLAIM_SHORT.get(x, x) for x in df.claim], xlim=(0, 1.06), xlabel='Support fraction', title='Method\nrobustness')
    ax.legend(fontsize=FONT['legend'], loc='lower right'); _style(ax)

def _best(ax):
    df = data['best_per_aspect'].copy()
    df['label'] = df.line.map(data['label_map']).fillna(df.line)
    short = ['Core hits', 'Expr. R₂', 'Clock TF', 'SE R₂', 'Lag R₂', 'RBP R₂', 'CLIP–APA']
    df['short'] = short
    y = np.arange(len(df)); ax.scatter(df.value, y, color=PALETTE['blue'], s=26, zorder=2)
    for i, row in enumerate(df.itertuples()):
        ax.annotate(row.label, (row.value, i), xytext=(-2 if row.value > 40 else 2, 3),
            ha='right' if row.value > 40 else 'left', textcoords='offset points', fontsize=FONT['annotation'])
    ax.set_xlim(-3, 66)
    ax.set_ylim(-.6, len(df)-.1)
    ax.set(yticks=y, yticklabels=df.short, xlabel='Best observed value', title='Best example\nby axis'); _style(ax)

def _drivers(ax):
    fields = ['KRAS', 'TP53', 'SMAD4', 'CDKN2A']
    d = data['drivers'].set_index('line').reindex(data['label_order'])
    codes = {'WT': 0, 'mut': 1, 'del': 2, 'nd': 3}
    vals = np.array([[codes[d.loc[line, field]] for line in d.index] for field in fields])
    colors = ['#F2F2F2', PALETTE['orange'], '#8BB9D4', '#FFFFFF']
    ax.imshow(vals, aspect='auto', cmap=mpl.colors.ListedColormap(colors), vmin=-.5, vmax=3.5)
    ax.set(xticks=range(len(d)), xticklabels=[data['label_map'][x] for x in d.index],
           yticks=range(len(fields)), yticklabels=fields, title='Driver annotations')
    ax.tick_params(axis='x', labelrotation=45, labelsize=FONT['tick'])
    ax.tick_params(axis='y', labelsize=FONT['tick'])
    for i, field in enumerate(fields):
        for j, line in enumerate(d.index):
            ax.text(j, i, d.loc[line, field], ha='center', va='center', fontsize=FONT['annotation'])


def _metric_heatmap(ax):
    cols = ['expr_R2','clockTF_OR','SE_R2','APA_med_amp','APAlag_R2','clipAPA_nRBP','RBP_R2','n_coreclock']
    labels = ['Expr R₂','Clock OR','SE R₂','APA amp','APA lag R₂','CLIP RBPs','RBP R₂','Core hits']
    raw = data['metrics'].set_index('line').reindex(data['label_order'])[cols].T
    z = raw.apply(lambda r: (r-r.mean())/(r.std(ddof=0) if r.std(ddof=0) else 1), axis=1)
    im = ax.imshow(z, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
    ax.set(xticks=range(raw.shape[1]), xticklabels=[data['label_map'][x] for x in raw.columns], yticks=range(len(labels)), yticklabels=labels, title='Scaled effect-size matrix')
    for i in range(raw.shape[0]):
        for j in range(raw.shape[1]):
            value=raw.iloc[i,j]
            if np.isfinite(value):
                annotation=f'{value:.2g}'
                # Retain two significant figures with room between adjacent cells.
                if annotation.startswith('0.'): annotation=annotation[1:]
                ax.text(j,i,annotation,ha='center',va='center',fontsize=FONT['annotation'], color='white' if abs(z.iloc[i,j]) > 1.3 else 'black')
    ax.tick_params(axis='x', labelrotation=45, labelsize=FONT['tick']); ax.tick_params(axis='y', labelsize=FONT['tick'])
    cb=ax.figure.colorbar(im, ax=ax, fraction=.03, pad=.02); cb.set_label('Within-metric z-score', fontsize=FONT['axis_title']); cb.ax.tick_params(labelsize=FONT['tick'])

def _complete(ax):
    p = phase_stats.pivot(index='layer', columns='line', values='n').reindex(LAYERS).reindex(columns=data['label_order'])
    im=ax.imshow(np.log10(p+1), aspect='auto', cmap='YlGnBu')
    ax.set(xticks=range(p.shape[1]), xticklabels=[data['label_map'][x] for x in p.columns], yticks=range(len(p)), yticklabels=p.index, title='Phase values available')
    ax.tick_params(axis='x', labelrotation=45, labelsize=FONT['tick']); ax.tick_params(axis='y', labelsize=FONT['tick'])
    for i in range(p.shape[0]):
        for j in range(p.shape[1]): ax.text(j,i,str(int(p.iloc[i,j])),ha='center',va='center',fontsize=FONT['annotation'],color='white' if np.log10(p.iloc[i,j]+1)>2.4 else 'black')
    cb=ax.figure.colorbar(im, ax=ax, fraction=.04,pad=.02); cb.set_label('log₁₀(n+1)',fontsize=FONT['axis_title']); cb.ax.tick_params(labelsize=FONT['tick'])

def _core(ax):
    genes = list(dict.fromkeys(core.gene.astype(str)))
    hit = core.pivot(index='gene',columns='line',values='is_rhythmic').reindex(genes).reindex(columns=data['label_order']).fillna(False).astype(int)
    phase = core.pivot(index='gene',columns='line',values='acrophase').reindex(genes).reindex(columns=data['label_order'])
    ax.imshow(hit,aspect='auto',cmap=mpl.colors.ListedColormap(['#F4F4F4',PALETTE['blue']]),vmin=0,vmax=1)
    ax.set(xticks=range(hit.shape[1]),xticklabels=[data['label_map'][x] for x in hit.columns],yticks=range(hit.shape[0]),yticklabels=hit.index,title='Core-clock BioCycle hits (lag, h)')
    ax.tick_params(axis='x',labelrotation=45,labelsize=FONT['tick']); ax.tick_params(axis='y',labelsize=FONT['tick'])
    for i in range(hit.shape[0]):
        for j in range(hit.shape[1]):
            if hit.iloc[i,j] and np.isfinite(phase.iloc[i,j]): ax.text(j,i,f'{phase.iloc[i,j]:.0f}',ha='center',va='center',fontsize=FONT['annotation'],color='white')

# Each panel is drawn inside a physical module-sized SubFigure. The same renderer is
# used for its individual export and its assembled position, so labels cannot borrow
# space from adjacent panels.
from matplotlib.ticker import MaxNLocator

_best_line = data['label_order'][0]
_small_specs = {
    '6G': ('expr_R2', 'Expression\npolarization', 'Axial R₂', 'expr_p'),
    '6H': ('SE_R2', 'Skipped-exon\npolarization', 'Axial R₂', 'SE_p'),
    '6I': ('APAlag_R2', 'APA–mRNA lag\nconcentration', 'Axial R₂', 'APAlag_p'),
    '6J': ('RBP_R2', 'RBP phase\norganization', 'Axial R₂', 'RBP_p'),
    'S17B': ('clipAPA_nRBP', 'CLIP–APA\nenrichment', 'Significant RBPs', None),
    'S17C': ('n_rhythmic_genes', 'Rhythmic expression\nburden', 'Rhythmic genes', None),
    'S17D': ('APA_med_amp', 'APA rhythmic\namplitude', 'APA amplitude', None),
    'S17E': ('n_coreclock', 'Core-clock\nrhythmic hits', 'Core-clock genes', None),
}
# key: (filename content, width in modules, height in modules)
PANEL_SPECS = {
    '6A': ('miapaca2_kronos_reporters',1,1),
    '6B': ('hs766t_kronos_reporters',1,1),
    '6C': ('panc0813_kronos_reporters',1,1),
    '6D': ('overall_alignment',1,1), '6E': ('rhythmic_expression_heatmap',3,2),
    '6F': ('phase_density',1,2), '6G': ('expression_polarization',1,1),
    '6H': ('skipped_exon_polarization',1,1), '6I': ('apa_mrna_lag_concentration',1,1),
    '6J': ('rbp_phase_organization',1,1), '6K': ('module_timing',2,1),
    '6L': ('alignment_summary',2,1),
    'S16A': ('tcga_transcriptome_pca',1,1), 'S16B': ('core_clock_tcga_percentiles',2,1),
    'S16C': ('composite_alignment_rank',1,1), 'S16D': ('method_robustness',1,1),
    'S16E': ('best_examples',1,1), 'S16F': ('driver_annotations',2,1),
    'S16G': ('scaled_effect_size_matrix',2,1), 'S16H': ('phase_value_availability',2,1),
    'S16I': ('core_clock_hits',4,1), 'S16J': ('panc1_expression_heatmap',2,1),
    'S16K': ('panc1005_expression_heatmap',2,1),
    'S17A': ('all_line_phase_density',4,8),
    'S17B': ('clip_apa_enrichment',1,1), 'S17C': ('rhythmic_expression_burden',1,1),
    'S17D': ('apa_amplitude',1,1), 'S17E': ('core_clock_hits',1,1),
}

def _phase_grid(container):
    # Two five-line blocks, four layers each. Give each polar chart a fixed
    # physical diameter and reserve space around every set of clock ticks.
    left, column_width = .065, .184
    top, row_height = .955, .115
    for r in range(8):
        block, layer_index = divmod(r, 4)
        layer = LAYERS[layer_index]
        cell_top = top-r*row_height
        container.text(.018, cell_top-.55*row_height, layer, rotation=90,
            ha='center', va='center', fontsize=FONT['strip'])
        for j, line in enumerate(data['label_order'][block*5:(block+1)*5]):
            x = left+j*column_width
            ax = container.add_axes([x+.155*column_width, cell_top-.816*row_height,
                                     .63*column_width, .504*row_height], projection='polar')
            sub = phase_density.query('line == @line and layer == @layer').sort_values('theta')
            row = phase_stats.query('line == @line and layer == @layer').iloc[0]
            if int(row.n) >= 3:
                ax.plot(sub.theta, sub.density, color=LAYER_COLORS[layer], lw=.8)
                ax.fill_between(sub.theta, 0, sub.density, color=LAYER_COLORS[layer], alpha=.16)
                for angle in (row.axis_phase_rad, row.opposite_axis_phase_rad):
                    ax.axvline(angle, color=PALETTE['orange'], lw=.6, ls='--')
            else:
                ax.text(.5,.5,f'n={int(row.n)}\ninsufficient', transform=ax.transAxes,
                    ha='center',va='center',fontsize=FONT['annotation'])
            ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
            ax.set_xticks([0,np.pi/2,np.pi,3*np.pi/2])
            ax.set_xticklabels(['0','6','12','18'],fontsize=FONT['tick'])
            ax.tick_params(pad=1)
            ax.set_yticklabels([])
            if layer_index == 0:
                container.text(x+.47*column_width, cell_top-.05*row_height,
                    data['label_map'][line], ha='center', va='top', fontsize=FONT['strip'])
    container.text(.5,.991,'Phase densities across retained cell lines',ha='center',va='top',fontsize=FONT['title'])

# Kronos reporter panels use the same raw-unit fits as the original analysis.
KRONOS_PANELS = {
    '6A': ('MiaPaca2', 'MIA PaCa-2'),
    '6B': ('Hs766t', 'Hs 766T'),
    '6C': ('Paca0813', 'Panc 08.13'),
}
KRONOS_COLORS = {'BMAL1': '#0072B2', 'PER2': '#D55E00'}

def _kronos_handles():
    from matplotlib.lines import Line2D
    return [Line2D([], [], color=KRONOS_COLORS[g], lw=1.25, label=g)
            for g in ('BMAL1', 'PER2')] + [
        Line2D([], [], color='0.4', marker='o', lw=0, markersize=2.5, alpha=.45, label='Data'),
        Line2D([], [], color='0.25', lw=1.25, label='Cosinor fit'),
        Line2D([], [], color='0.4', lw=.65, ls='--', label='MESOR'),
        Line2D([], [], color='0.4', lw=.7, ls=':', label='Acrophase')]

def _kronos_panel(container, key, assembled=False):
    cell_line, label = KRONOS_PANELS[key]
    ax = container.add_axes([.235, .27 if assembled else .43, .735, .55 if assembled else .39])
    for gene in ('BMAL1', 'PER2'):
        d = kronos_observed.query('cell_line == @cell_line and gene == @gene')
        c = kronos_curves.query('cell_line == @cell_line and gene == @gene')
        s = kronos_summary.query('cell_line == @cell_line and gene == @gene').iloc[0]
        color = KRONOS_COLORS[gene]
        mesor_z = (s.mesor - d.signal.mean()) / d.signal.std(ddof=0)
        ax.scatter(d.plot_time_h, d.signal_zscore, s=3.5, color=color, alpha=.22,
                   linewidths=0, zorder=2)
        ax.plot(c.plot_time_h, c.fitted_zscore, color=color, lw=1.25, zorder=4)
        ax.axhline(mesor_z, color=color, lw=.65, ls='--', alpha=.65, zorder=1)
        peak = s.acrophase_h + KRONOS_PERIOD_H * np.ceil(
            (d.time_h.min() - s.acrophase_h) / KRONOS_PERIOD_H)
        if peak <= d.time_h.max():
            ax.axvline(peak - s.display_time_offset_h, color=color, lw=.7, ls=':', alpha=.8, zorder=1)
    ax.set_xlim(0, 60)
    ax.set_xticks(np.arange(0, 61, 12))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    # Ensure locator ticks outside the data limits do not enter the bounds audit.
    lo, hi = ax.get_ylim()
    ax.set_yticks([v for v in ax.get_yticks() if lo <= v <= hi])
    ax.set_xlabel('Time in window (h)', labelpad=2)
    if not assembled or key == '6A':
        ax.set_ylabel('Signal (z-score)', labelpad=2)
    ax.tick_params(pad=2, length=2.5)
    container.text(.5,.965,label,ha='center',va='top',fontsize=FONT['title'],weight='bold')
    if not assembled:
        handles = _kronos_handles()
        # Column-major legend order makes the rows reporters, data/fit, and reference lines.
        handles = [handles[i] for i in (0,2,4,1,3,5)]
        container.legend(handles=handles,loc='lower center',bbox_to_anchor=(.5,.005),
            ncol=2,frameon=False,fontsize=FONT['legend'],handlelength=1.4,
            columnspacing=.7,handletextpad=.4,labelspacing=.25)


def _s16_tcga_panel(container, key):
    from matplotlib.lines import Line2D
    from matplotlib.legend_handler import HandlerTuple
    colors={cell:mpl.colormaps['tab10'](i) for i,cell in enumerate(S16_TCGA['cell_order'])}
    if key == 'S16A':
        ax=container.add_axes([.25,.29,.70,.49])
        scores=S16_TCGA['pca_scores']; variance=S16_TCGA['pca_variance']
        patient=scores.loc[scores.source.eq('TCGA')]
        lines=scores.loc[scores.source.eq('Cell line')]
        ax.scatter(patient.PC1,patient.PC2,s=4,c='.72',alpha=.55,linewidths=0)
        for cell in S16_TCGA['cell_order']:
            z=lines.loc[lines.cell.eq(cell)]
            ax.scatter(z.PC1,z.PC2,s=13,c=[colors[cell]],edgecolor='white',linewidth=.3,zorder=3)
        ax.set_xlabel(f'PC1 ({100*variance[0]:.1f}%)',fontsize=FONT['axis_title'],labelpad=1)
        ax.set_ylabel(f'PC2 ({100*variance[1]:.1f}%)',fontsize=FONT['axis_title'],labelpad=1)
        ax.tick_params(labelsize=FONT['tick'],length=2,pad=1)
        ax.xaxis.set_major_locator(MaxNLocator(3)); ax.yaxis.set_major_locator(MaxNLocator(3))
        for direction in ('x','y'):
            lo,hi=sorted(getattr(ax,f'get_{direction}lim')())
            ticks=getattr(ax,f'get_{direction}ticks')()
            getattr(ax,f'set_{direction}ticks')([t for t in ticks if lo<=t<=hi])
        gray=Line2D([],[],linestyle='',marker='o',markersize=3,color='.65')
        multiple=tuple(Line2D([],[],linestyle='',marker='o',markersize=3,color=colors[c]) for c in S16_TCGA['cell_order'][:3])
        container.legend([gray,multiple],['TCGA','Cell lines'],handler_map={tuple:HandlerTuple(ndivide=None,pad=0)},
            loc='lower center',bbox_to_anchor=(.52,.005),ncol=2,fontsize=FONT['legend'],frameon=False,
            columnspacing=.7,handlelength=1.5,handletextpad=.3)
        title='Transcriptome PCA'
    else:
        # Keep the established S16 cell-line rank order; colors identify PCA points.
        order=[data['label_map'][line] for line in data['label_order']]
        values=S16_TCGA['clock_percentiles'].reindex(columns=order)
        assert values.notna().all().all(), 'Clock percentile labels must match the retained lines'
        ax=container.add_axes([.15,.32,.71,.58])
        im=ax.imshow(values,aspect='auto',cmap='RdBu_r',vmin=0,vmax=100,interpolation='nearest')
        ax.set_yticks(range(len(values)),values.index)
        ax.set_xticks(range(len(order)),order,rotation=45,ha='right',rotation_mode='anchor')
        ax.tick_params(labelsize=FONT['tick'],length=2,pad=1)
        for label in ax.get_xticklabels():label.set_color(colors[label.get_text()])
        cax=container.add_axes([.88,.32,.018,.58]); cb=container.colorbar(im,cax=cax)
        cb.set_ticks([0,50,100]);cb.ax.tick_params(labelsize=FONT['tick'],length=2,pad=1)
        cb.set_label('TCGA percentile',fontsize=FONT['axis_title'],labelpad=1)
        title='Core-clock expression'
    container.text(.5,.965,title,ha='center',va='top',fontsize=FONT['title'],weight='bold')


def _s16_core_clock_grid(ax):
    # Transposition keeps all existing values and permits a full-width single row.
    genes=list(dict.fromkeys(core.gene.astype(str)))
    hit=core.pivot(index='gene',columns='line',values='is_rhythmic').reindex(genes).reindex(columns=data['label_order']).fillna(False).astype(int).T
    phase=core.pivot(index='gene',columns='line',values='acrophase').reindex(genes).reindex(columns=data['label_order']).T
    ax.imshow(hit,aspect='auto',cmap=mpl.colors.ListedColormap(['#F4F4F4',PALETTE['blue']]),vmin=0,vmax=1)
    ax.set_xticks(range(len(genes)),genes,rotation=45,ha='right',rotation_mode='anchor')
    ax.set_yticks(range(len(hit)),[data['label_map'][line] for line in hit.index])
    ax.tick_params(labelsize=FONT['tick'],length=2,pad=1)
    for i in range(hit.shape[0]):
        for j in range(hit.shape[1]):
            if hit.iloc[i,j] and np.isfinite(phase.iloc[i,j]):
                ax.text(j,i,f'{phase.iloc[i,j]:.0f}',ha='center',va='center',fontsize=FONT['annotation'],color='white')


def _render_panel(container, key, assembled=False):
    if key in ('S16A','S16B'):
        _s16_tcga_panel(container,key)
        return
    if key in KRONOS_PANELS:
        _kronos_panel(container, key, assembled=assembled)
        return
    if key == 'S17A':
        _phase_grid(container)
        return
    rect = [.39,.24,.56,.51]
    title = ''
    if key in _small_specs:
        title = _small_specs[key][1]
    elif key == '6D': rect=[.34,.24,.62,.51]; title='Alignment to cohort\nconclusions'
    elif key == '6E': rect=[.075,.10,.825,.73]; title='Hs 766T: rhythmic expression\nrow-scaled, sorted by BioCycle lag'
    elif key == '6F': rect=[.16,.36,.68,.48]; title='Hs 766T\nPhase densities'
    elif key == '6K': rect=[.30,.34,.56,.43]; title='Rhythmic fraction in curated modules'
    elif key == '6L': rect=[.15,.24,.81,.53]; title='Cell-line alignment phenotypes'
    elif key == 'S16C': rect=[.38,.24,.57,.51]; title='Composite\nalignment rank'
    elif key == 'S16D': rect=[.44,.30,.51,.47]; title='Method\nrobustness'
    elif key == 'S16E': rect=[.34,.24,.62,.51]; title='Best example\nper axis'
    elif key == 'S16F': rect=[.16,.36,.81,.41]; title='Driver annotations'
    elif key == 'S16G': rect=[.23,.35,.66,.44]; title='Scaled effect-size matrix'
    elif key == 'S16H': rect=[.18,.36,.72,.41]; title='Phase values available by layer'
    elif key == 'S16I': rect=[.115,.28,.855,.57]; title='Core-clock BioCycle hits (lag, h)'
    elif key in ('S16J','S16K'): rect=[.14,.27,.73,.47]; title=('PANC-1' if key=='S16J' else 'Panc 10.05')+': rhythmic expression\nrow-scaled, sorted by BioCycle lag'
    ax=container.add_axes(rect,projection='polar' if key=='6F' else None)
    if key in _small_specs:
        value, _, xlabel, pcol = _small_specs[key]
        _lollipop(ax,value,'',xlabel,pcol)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=3,integer=key in ('S17B','S17C','S17E')))
    elif key=='6D': _overall(ax); ax.set_xticks([0,.5,1])
    elif key=='6E': _heatmap(ax,heatmaps[_best_line],'')
    elif key=='6F':
        _combined_density(ax,_best_line)
        handles, labels=ax.get_legend_handles_labels()
        ax.get_legend().remove()
        container.legend(handles,labels,loc='lower center',bbox_to_anchor=(.5,.13),
            fontsize=FONT['legend'],frameon=False,handlelength=1.1,labelspacing=.4)
        container.text(.5,.10,'Dashed: axial orientation',ha='center',fontsize=FONT['annotation'])
    elif key=='6K': _module_heatmap(ax)
    elif key=='6L': _summary(ax); ax.margins(y=.18)
    elif key=='S16C': _rank(ax); ax.xaxis.set_major_locator(MaxNLocator(3))
    elif key=='S16D':
        _method(ax)
        short={'Expr. polarization':'Expression','Splicing polarization':'Splicing','APA–mRNA lag':'APA–mRNA','RBP phase':'RBP'}
        ax.set_yticklabels([short.get(t.get_text(),t.get_text()) for t in ax.get_yticklabels()])
        ax.set_xticks([0,.5,1])
        handles,labels=ax.get_legend_handles_labels();ax.get_legend().remove()
        container.legend(handles,labels,loc='lower center',bbox_to_anchor=(.52,.005),
            ncol=2,frameon=False,fontsize=FONT['legend'],columnspacing=.7,handletextpad=.3)
    elif key=='S16E': _best(ax); ax.set_xlabel('Observed value');ax.set_xticks([0,30,60])
    elif key=='S16F':
        _drivers(ax)
        container.text(.5,.015,'WT: reported WT; mut: variant; del: deletion; nd: unresolved',
            ha='center',va='bottom',fontsize=FONT['annotation'])
    elif key=='S16G': _metric_heatmap(ax)
    elif key=='S16H':
        _complete(ax)
        ax.set_yticklabels(['Expression', 'SE', 'APA', 'RBP'],fontsize=FONT['tick'])
        ax.set_xticks(np.arange(-.5,10,1),minor=True)
        ax.set_yticks(np.arange(-.5,4,1),minor=True)
        ax.grid(which='minor',color='white',linewidth=.5)
        ax.tick_params(which='minor',length=0)
    elif key=='S16I': _s16_core_clock_grid(ax)
    elif key in ('S16J','S16K'):
        _heatmap(ax,heatmaps['PANC1_Dx' if key=='S16J' else 'PANC1005_Dexa'],'')
    if key in ('6K','S16F','S16G','S16H','S16I','S16J','S16K'):
        for label in ax.get_xticklabels():
            label.set_rotation(45);label.set_ha('right');label.set_rotation_mode('anchor')
    # Remove locator ticks outside the displayed range before export auditing.
    for panel_ax in container.axes:
        if panel_ax.name == 'polar':
            continue
        for direction in ('x', 'y'):
            bounds = sorted(getattr(panel_ax, f'get_{direction}lim')())
            ticks = getattr(panel_ax, f'get_{direction}ticks')()
            kept = [t for t in ticks if bounds[0]-1e-10 <= t <= bounds[1]+1e-10]
            if len(kept) != len(ticks):
                getattr(panel_ax, f'set_{direction}ticks')(kept)
    ax.set_title('')
    container.text(.5,.965,title,ha='center',va='top',fontsize=FONT['title'],weight='bold',linespacing=1.05)


def _individual(key):
    content,w,h=PANEL_SPECS[key]
    fig=_fig(w,h)
    container=fig.subfigures(1,1)
    _render_panel(container,key)
    save_figure(fig,f'Figure{key}_{content}',w,h,tags=[(key[-1],1.5/(w*PANEL_MM),1-1.5/(h*PANEL_MM))])


def _assembled(stem, height, positions):
    fig=_fig(4,height)
    grid=fig.add_gridspec(height,4,left=0,right=1,bottom=0,top=1,wspace=0,hspace=0)
    tags=[]
    for key,row,col in positions:
        _,w,h=PANEL_SPECS[key]
        container=fig.add_subfigure(grid[row:row+h,col:col+w])
        _render_panel(container,key,assembled=True)
        tags.append((key[-1],(col+1.5/PANEL_MM)/4,1-(row+1.5/PANEL_MM)/height))
    if stem == 'Figure6_assembled_cellline_validation':
        fig.legend(handles=_kronos_handles(),loc='lower center',
            bbox_to_anchor=(.375,1-1/height+.01/height),ncol=6,frameon=False,
            fontsize=FONT['legend'],handlelength=1.4,columnspacing=1.0,handletextpad=.4)
    save_figure(fig,stem,4,height,tags=tags)


In [ ]:
# Cell 11: Figure 6 panels and assembly
# Individual Figure 6 panels and their assembly, including the three Kronos reporter panels.
for key in ['6A','6B','6C','6D','6E','6F','6G','6H','6I','6J','6K','6L']:
    _individual(key)
_assembled('Figure6_assembled_cellline_validation',5,
    [('6A',0,0),('6B',0,1),('6C',0,2),('6D',0,3),('6E',1,0),('6F',1,3),('6G',3,0),('6H',3,1),
     ('6I',3,2),('6J',3,3),('6K',4,0),('6L',4,2)])


In [ ]:
# Cell 12: Supplementary Figure 16 panels and five-row assembly
for key in ['S16A','S16B','S16C','S16D','S16E','S16F','S16G','S16H','S16I','S16J','S16K']:
    _individual(key)
_assembled('FigureS16_assembled_validation_details',5,
    [('S16A',0,0),('S16B',0,1),('S16C',0,3),
     ('S16D',1,0),('S16E',1,1),('S16F',1,2),
     ('S16G',2,0),('S16H',2,2),('S16I',3,0),('S16J',4,0),('S16K',4,2)])


In [ ]:
# Cell 13: Supplementary Figure 17 panels and assembly
# The complete two-block density grid is S17A; B–E occupy the last row.
for key in ['S17A','S17B','S17C','S17D','S17E']:
    _individual(key)
_assembled('FigureS17_assembled_cellline_comparisons',9,
    [('S17A',0,0),('S17B',8,0),('S17C',8,1),('S17D',8,2),('S17E',8,3)])


## 4. Export validation and provenance

In [ ]:
# Cell 14: Validate exports and record input hashes and software versions
export_manifest = pd.DataFrame(EXPORT_MANIFEST)
assert len(export_manifest) == 31, f'Expected 31 panel/assembly pairs, got {len(export_manifest)}'
assert export_manifest['stem'].is_unique
assert export_manifest['minimum_text_pt'].min() >= 7
assert not export_manifest['png_has_panel_labels'].any()
assert export_manifest['pdf_has_panel_labels'].all()
for column in ('png', 'pdf'):
    assert all((FIGURE_DIR / name).is_file() and (FIGURE_DIR / name).stat().st_size > 0
               for name in export_manifest[column])
export_manifest.to_csv(TABLE_DIR / 'publication_export_manifest.csv', index=False)
pd.DataFrame(INPUT_MANIFEST.values()).sort_values('input_path').to_csv(
    TABLE_DIR / 'publication_input_manifest.csv', index=False)
pd.DataFrame([dict(role=key, font_size_pt=value, family=FONT_FAMILY)
              for key, value in FONT.items()]).to_csv(TABLE_DIR / 'publication_font_configuration.csv', index=False)
versions = {'python': platform.python_version(), 'platform': platform.platform(),
            'elapsed_seconds': round(time.time() - START_TIME, 2), 'rng_seed': RNG_SEED}
for package in ('numpy', 'pandas', 'scipy', 'matplotlib', 'ipython', 'openpyxl', 'scikit-learn'):
    versions[package] = importlib.metadata.version(package)
(TABLE_DIR / 'publication_session_info.json').write_text(json.dumps(versions, indent=2) + '\n')
print('Exported 31 PNG/PDF pairs; PNGs omit panel letters and PDFs include them.')
print('Processed input files recorded:', len(INPUT_MANIFEST))
display(export_manifest[['stem', 'width_mm', 'height_mm', 'minimum_text_pt', 'outside_text']])
if export_manifest['outside_text'].astype(bool).any():
    raise AssertionError('Inspect out-of-page text reported in publication_export_manifest.csv.')
print('Validation complete: minimum font size, labels, export count, and page bounds passed.')
